In [4]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import json
from pathlib import Path
from collections import Counter
import math
import statistics
import pandas as pd

## Humans vs AgentSLR - Gantt Chart

In [5]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import os
import numpy as np

os.makedirs("../assets/figures", exist_ok=True)

FONT_FAMILY = "Times New Roman, Times, serif"
TICK_FONT_SIZE = 19
TITLE_FONT_SIZE = 19

COLOR_FULLTEXT_SCREENING = "#6BAED6"
COLOR_DATA_EXTRACTION = "#E38B48"
COLOR_TITLE_ABSTRACT = "#B7D3C6"
COLOR_PDF_CONVERSION = "#C9A3A3"

In [6]:
df_time_stats = pd.read_csv("../data/agentslr/evals/stats/time_per_article_minutes.csv",)

In [7]:
# Average articles passed through each stage.
t_and_a = 9132
full_text = 1102
extraction = 395

In [8]:
average_df = df_time_stats.groupby('Model').mean(numeric_only=True).reset_index()

time_df = (
    average_df
    .assign(
        **{
            'Title & Abstract Screening': lambda d: ((d['Title & Abstract Screening'] * t_and_a) + (d['Download and Harvest'] * t_and_a)) / 60,
            'Full-text Screening': lambda d: d['Full-text Screening'] * full_text / 60,
            'PDF-to-MD Conversion': lambda d: d['PDF-to-MD Conversion'] * t_and_a / 60,
            'Data Extraction': lambda d: (
                d['Model Extraction']  * extraction +
                d['Parameter Extraction']  * extraction +
                d['Outbreak Extraction']  * extraction
            ) / 60
        }
    )
    [['Model', 'Title & Abstract Screening', 'Full-text Screening', 'PDF-to-MD Conversion', 'Data Extraction']]
    .assign(Total=lambda d: d.iloc[:, 1:].sum(axis=1))
)

In [9]:
r = time_df[time_df["Model"] == "oss"].to_dict(orient="records")[0]

order = ["Title & Abstract Screening", "PDF-to-MD Conversion", "Full-text Screening", "Data Extraction"]
colors = {
    "Title & Abstract Screening": COLOR_TITLE_ABSTRACT,
    "PDF-to-MD Conversion": COLOR_PDF_CONVERSION,
    "Full-text Screening": COLOR_FULLTEXT_SCREENING,
    "Data Extraction": COLOR_DATA_EXTRACTION,
}

s = 0.0
agentslr_gantt = []
for t in order:
    agentslr_gantt.append({"task": t, "start": s, "duration": r[t], "color": colors[t]})
    s += r[t]

In [10]:
# AGENTSLR GANTT CHART
[{'task': 'Title & Abstract Screening',
  'start': 0.0,
  'duration': 3.1976788766666524,
  'color': '#B7D3C6'},
 {'task': 'PDF-to-MD Conversion',
  'start': 3.1976788766666524,
  'duration': 2.790333333333328,
  'color': '#C9A3A3'},
 {'task': 'Full-text Screening',
  'start': 5.98801220999998,
  'duration': 0.6211826301388883,
  'color': '#6BAED6'},
 {'task': 'Data Extraction',
  'start': 6.609194840138869,
  'duration': 13.39549433611111,
  'color': '#E38B48'}]

[{'task': 'Title & Abstract Screening',
  'start': 0.0,
  'duration': 3.1976788766666524,
  'color': '#B7D3C6'},
 {'task': 'PDF-to-MD Conversion',
  'start': 3.1976788766666524,
  'duration': 2.790333333333328,
  'color': '#C9A3A3'},
 {'task': 'Full-text Screening',
  'start': 5.98801220999998,
  'duration': 0.6211826301388883,
  'color': '#6BAED6'},
 {'task': 'Data Extraction',
  'start': 6.609194840138869,
  'duration': 13.39549433611111,
  'color': '#E38B48'}]

In [11]:
human_perg_total_hours = {
    "Title & Abstract Screening": ((45 / 60) * t_and_a)/60,
    "Full-text Screening": ((240 / 60) * full_text)/60,
    "Data Extraction": ((1800 / 60) * extraction)/60,
}

order = ["Title & Abstract Screening", "Full-text Screening", "Data Extraction"]
colors = {
    "Title & Abstract Screening": COLOR_TITLE_ABSTRACT,
    "Full-text Screening": COLOR_FULLTEXT_SCREENING,
    "Data Extraction": COLOR_DATA_EXTRACTION,
}

perg_gantt = []
s = 0.0
for t in order:
    perg_gantt.append({"task": t, "start": s, "duration": human_perg_total_hours[t], "color": colors[t]})
    s += human_perg_total_hours[t]

In [12]:
### AGENTSLR GANTT CHART
[{'task': 'Title & Abstract Screening',
  'start': 0.0,
  'duration': 3.1976788766666524,
  'color': '#B7D3C6'},
 {'task': 'PDF-to-MD Conversion',
  'start': 3.1976788766666524,
  'duration': 2.790333333333328,
  'color': '#C9A3A3'},
 {'task': 'Full-text Screening',
  'start': 5.98801220999998,
  'duration': 0.6211826301388883,
  'color': '#6BAED6'},
 {'task': 'Data Extraction',
  'start': 6.609194840138869,
  'duration': 13.39549433611111,
  'color': '#E38B48'}]

### Humans (PERG GANTT)

[{'task': 'Title & Abstract Screening',
  'start': 0.0,
  'duration': 114.15,
  'color': '#B7D3C6'},
 {'task': 'Full-text Screening',
  'start': 114.15,
  'duration': 73.46666666666667,
  'color': '#6BAED6'},
 {'task': 'Data Extraction',
  'start': 187.61666666666667,
  'duration': 197.5,
  'color': '#E38B48'}]

[{'task': 'Title & Abstract Screening',
  'start': 0.0,
  'duration': 114.15,
  'color': '#B7D3C6'},
 {'task': 'Full-text Screening',
  'start': 114.15,
  'duration': 73.46666666666667,
  'color': '#6BAED6'},
 {'task': 'Data Extraction',
  'start': 187.61666666666667,
  'duration': 197.5,
  'color': '#E38B48'}]

In [13]:
perg_total = sum(t["duration"] for t in perg_gantt)
agentslr_total = sum(t["duration"] for t in agentslr_gantt)

fig = make_subplots(
    rows=1, cols=2,
    column_widths=[0.64, 0.36],
    horizontal_spacing=0.08,
)

X_HUMANS = 0.00
X_AGENT_L = 0.20
X_AGENT_R = 0.00
BAR_W = 0.14

def add_rounded_top_segment(
    fig,
    *,
    x_center,
    y0,
    y1,
    color,
    row,
    col,
    bar_w,
    rx_frac=0.32,
    ry_frac=0.22,
    arc_pts=40,
    squircle_n=6,
):
    if y1 <= y0:
        return
    xl = x_center - bar_w / 2
    xr = x_center + bar_w / 2
    h = y1 - y0
    rx = bar_w * rx_frac
    ry = min(h * ry_frac, h)
    if rx <= 0 or ry <= 0 or h <= 0 or (y1 - ry) < y0:
        xs = [xl, xr, xr, xl, xl]
        ys = [y0, y0, y1, y1, y0]
        fig.add_trace(
            go.Scatter(
                x=xs, y=ys,
                mode="lines",
                line=dict(width=0),
                fill="toself",
                fillcolor=color,
                hoverinfo="skip",
                showlegend=False,
            ),
            row=row, col=col
        )
        return
    p = 2.0 / float(squircle_n)
    t = np.linspace(0.0, np.pi / 2.0, arc_pts)
    ct = np.cos(t) ** p
    st = np.sin(t) ** p
    tr_x = (xr - rx) + rx * ct
    tr_y = (y1 - ry) + ry * st
    tl_x = (xl + rx) - rx * ct
    tl_y = (y1 - ry) + ry * st
    xs = np.concatenate([
        [xl, xr],
        [xr],
        tr_x,
        [xl + rx],
        tl_x[::-1],
        [xl],
        [xl],
    ])
    ys = np.concatenate([
        [y0, y0],
        [y1 - ry],
        tr_y,
        [y1],
        tl_y[::-1],
        [y1 - ry],
        [y0],
    ])
    fig.add_trace(
        go.Scatter(
            x=xs, y=ys,
            mode="lines",
            line=dict(width=0),
            fill="toself",
            fillcolor=color,
            hoverinfo="skip",
            showlegend=False,
        ),
        row=row, col=col
    )

def add_stack_except_data_extraction(gantt, x, row, col):
    for task in gantt:
        if task["task"] == "Data Extraction":
            continue
        fig.add_trace(
            go.Bar(
                x=[x],
                y=[task["duration"]],
                base=task["start"],
                width=BAR_W,
                marker=dict(color=task["color"]),
                showlegend=False,
                hovertemplate=f'{task["task"]}<br>Duration: {task["duration"]:.2f}h<extra></extra>',
            ),
            row=row, col=col
        )

def add_orange_rounded(gantt, x, row, col):
    task = next(t for t in gantt if t["task"] == "Data Extraction")
    y0 = task["start"]
    y1 = task["start"] + task["duration"]
    add_rounded_top_segment(
        fig,
        x_center=x,
        y0=y0,
        y1=y1,
        color=task["color"],
        row=row,
        col=col,
        bar_w=BAR_W,
    )

add_stack_except_data_extraction(perg_gantt, X_HUMANS, 1, 1)
add_stack_except_data_extraction(agentslr_gantt, X_AGENT_L, 1, 1)
add_stack_except_data_extraction(agentslr_gantt, X_AGENT_R, 1, 2)

add_orange_rounded(perg_gantt, X_HUMANS, 1, 1)
add_orange_rounded(agentslr_gantt, X_AGENT_L, 1, 1)
add_orange_rounded(agentslr_gantt, X_AGENT_R, 1, 2)

legend_items = [
    ("Data Extraction", COLOR_DATA_EXTRACTION),
    ("Full-text Screening", COLOR_FULLTEXT_SCREENING),
    ("PDF-to-MD Conversion*", COLOR_PDF_CONVERSION),
    ("Title & Abstract Screening", COLOR_TITLE_ABSTRACT),
]
for name, color in legend_items:
    fig.add_trace(
        go.Bar(
            x=[None], y=[None],
            name=name,
            marker=dict(color=color),
            showlegend=True
        ),
        row=1, col=1
    )

common_axis = dict(
    showgrid=False,
    zeroline=False,
    showline=True,
    linecolor="black",
    linewidth=1,
    ticks="outside",
    tickcolor="black",
    tickfont=dict(family=FONT_FAMILY, size=TICK_FONT_SIZE, color="black"),
)

fig.update_yaxes(
    title_text="Time (hours)",
    title_font=dict(family=FONT_FAMILY, size=TITLE_FONT_SIZE, color="black"),
    range=[0, max(perg_total, agentslr_total) * 1.03],
    dtick=50,
    **common_axis,
    row=1, col=1
)
fig.update_yaxes(
    range=[0, 10],
    title_text="",
    **common_axis,
    row=1, col=2
)

fig.update_xaxes(
    range=[-0.2, 0.45],
    tickmode="array",
    tickvals=[X_HUMANS, X_AGENT_L],
    ticktext=["Humans", "AgentSLR"],
    **common_axis,
    row=1, col=1
)
fig.update_xaxes(
    range=[-0.2, 0.2],
    tickmode="array",
    tickvals=[X_AGENT_R],
    ticktext=["AgentSLR"],
    **common_axis,
    row=1, col=2
)

fig.update_layout(
    height=500,
    width=600,
    barmode="stack",
    plot_bgcolor="white",
    paper_bgcolor="white",
    font=dict(family=FONT_FAMILY, size=TICK_FONT_SIZE, color="black"),
    margin=dict(t=12, b=42, l=48, r=6),
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.07,
        xanchor="center",
        x=0.5,
        font=dict(family=FONT_FAMILY, size=TICK_FONT_SIZE),
        bgcolor="rgba(255,255,255,0)",
        borderwidth=0
    ),
)

fig.update_annotations(font=dict(family=FONT_FAMILY, size=TITLE_FONT_SIZE, color="black"))

# fig.write_image("../assets/figures/timeline_stats/time_stats_two_panel_magnified.pdf", scale=1)
# fig.write_image("../assets/figures/timeline_stats/time_stats_raw.svg", scale=1)

fig.show()

## Article Screening GPT-OSS Ablation

In [14]:
import pandas as pd

df = pd.read_csv("../data/agentslr/evals/article_screening/gpt_oss_article_screening_ablations.csv")

data = {
    r.pathogen: {
        "precision": (
            round(r.precision_ai4epi_abstract_conditioned, 3),
            round(r.precision_fulltext_direct, 3),
            round(r.precision_perg_conditioned, 3),
        ),
        "recall": (
            round(r.recall_ai4epi_abstract_conditioned, 3),
            round(r.recall_fulltext_direct, 3),
            round(r.recall_perg_conditioned, 3),
        ),
        "macro_f1": (
            round(r.f1_ai4epi_abstract_conditioned, 3),
            round(r.f1_fulltext_direct, 3),
            round(r.f1_perg_conditioned, 3),
        ),
    }
    for r in df.itertuples(index=False)
}

In [15]:
data

{'Marburg': {'precision': (0.746, 0.644, 0.775),
  'recall': (0.762, 0.818, 0.828),
  'macro_f1': (0.753, 0.695, 0.799)},
 'Ebola': {'precision': (0.731, 0.667, 0.862),
  'recall': (0.839, 0.928, 0.972),
  'macro_f1': (0.773, 0.72, 0.909)},
 'Lassa': {'precision': (0.786, 0.713, 0.832),
  'recall': (0.781, 0.912, 0.941),
  'macro_f1': (0.783, 0.767, 0.877)},
 'SARS': {'precision': (0.709, 0.636, 0.796),
  'recall': (0.852, 0.905, 0.954),
  'macro_f1': (0.758, 0.677, 0.855)},
 'Zika': {'precision': (0.656, 0.645, 0.805),
  'recall': (0.788, 0.849, 0.914),
  'macro_f1': (0.692, 0.674, 0.849)},
 'MERS': {'precision': (0.758, 0.693, 0.828),
  'recall': (0.83, 0.948, 0.96),
  'macro_f1': (0.789, 0.764, 0.882)},
 'Nipah': {'precision': (0.866, 0.74, 0.892),
  'recall': (0.842, 0.881, 0.9),
  'macro_f1': (0.853, 0.791, 0.896)},
 'Overall': {'precision': (0.75, 0.677, 0.827),
  'recall': (0.813, 0.892, 0.924),
  'macro_f1': (0.772, 0.727, 0.867)}}

In [16]:
import numpy as np
import plotly.graph_objects as go

FONT_FAMILY = "Times New Roman, Times, serif"
TICK_FONT_SIZE = 22
TITLE_FONT_SIZE = 22

# COLOR_TWO_STAGE = "#627D87"
# COLOR_PERG_FILTERED = "#9ABFD4"
# COLOR_DIRECT = "#2B7A9E"

COLOR_TWO_STAGE = "#8AC1E1"   # base
COLOR_DIRECT = "#3F86B5"      # darker
COLOR_PERG_FILTERED = "#1F5F8C"  # noticeably darker


n_trials = {k: 200 for k in data.keys()}

METRIC_ALIASES = {
    "prec": "precision",
    "precision": "precision",
    "rec": "recall",
    "recall": "recall",
    "f1": "macro_f1",
    "macro": "macro_f1",
    "macro_f1": "macro_f1",
    "micro": "micro_f1",
    "micro_f1": "micro_f1",
}

METRIC_LABELS = {
    "precision": "Precision",
    "recall": "Recall",
    "macro_f1": "Macro F1",
    "micro_f1": "Micro F1"
}

def normalize_metric_key(metric):
    return METRIC_ALIASES[metric.strip().lower()]

def bootstrap_ci_proportion(p, n, n_boot=10000, alpha=0.05, rng=None):
    p = float(np.clip(p, 0.0, 1.0))
    n = int(n)
    s = int(np.clip(np.round(p * n), 0, n))
    if rng is None:
        rng = np.random.default_rng(0)
    phat = rng.binomial(n, s / n, size=n_boot) / n
    return np.quantile(phat, alpha / 2.0), np.quantile(phat, 1.0 - alpha / 2.0)

In [17]:
LABEL_PAD = 0.01

def plot_metric(metric="recall", title=None, save_path_png=None, save_path_pdf=None, n_boot=10000, ci=95, seed=7):
    metric_key = normalize_metric_key(metric)
    pathogens = list(data.keys())
    n = len(pathogens)

    two_stage = np.array([data[p][metric_key][0] for p in pathogens])
    direct    = np.array([data[p][metric_key][1] for p in pathogens])
    perg      = np.array([data[p][metric_key][2] for p in pathogens])

    alpha = 1.0 - ci / 100.0
    rng = np.random.default_rng(seed)

    def ci_fn(arr):
        lo, hi = zip(*[
            bootstrap_ci_proportion(v, n_trials.get(p, 200), n_boot, alpha, rng)
            for v, p in zip(arr, pathogens)
        ])
        return np.array(lo), np.array(hi)

    lo_two, hi_two = ci_fn(two_stage)
    lo_dir, hi_dir = ci_fn(direct)
    lo_perg, hi_perg = ci_fn(perg)

    w = 0.25
    step = w + 0.02
    offsets = [-step, 0.0, step]
    xs = np.arange(n, dtype=float)

    bar_specs = [
        (two_stage,  hi_two,  lo_two,  offsets[0], COLOR_TWO_STAGE,    "AI Screen (Abstract) → AI Screen (Full-text)"),
        (direct,     hi_dir,  lo_dir,  offsets[1], COLOR_DIRECT,       "AI Screen (Direct Full-text)"),
        (perg,       hi_perg, lo_perg, offsets[2], COLOR_PERG_FILTERED, "Human Screen (Abstract) → AI Screen (Full-text)"),
    ]

    fig = go.Figure()

    for vals, hi, lo, offset, color, name in bar_specs:
        fig.add_bar(
            x=xs + offset, y=vals, width=w,
            marker=dict(color=color, line=dict(width=0)),
            error_y=dict(type="data", array=hi - vals, arrayminus=vals - lo,
                         symmetric=False, thickness=0.8, width=3, color="black"),
            name=name,
        )

    for i in range(n):
        group_vals = [two_stage[i], direct[i], perg[i]]
        group_hi   = [hi_two[i],    hi_dir[i],  hi_perg[i]]
        best_idx   = int(np.argmax(group_vals))
        for g, (val, hi_val, offset) in enumerate(zip(group_vals, group_hi, offsets)):
            is_best = (g == best_idx)
            fig.add_annotation(
                x=xs[i] + offset,
                y=hi_val + LABEL_PAD,
                text=f"<b>{val:.2f}</b>" if is_best else f"{val:.2f}",
                showarrow=False,
                font=dict(family=FONT_FAMILY, size=18,
                          color="#111111" if is_best else "#666666"),
                xanchor="center", yanchor="bottom",
            )

    min_off = min(offsets)
    max_off = max(offsets)

    pad = 0.075
    x_left  = xs[0]  + min_off - w/2 - pad
    x_right = xs[-1] + max_off + w/2 + pad

    fig.update_xaxes(range=[x_left, x_right])
    
    fig.update_layout(
        height=340,
        width=1200,
        margin=dict(t=0, b=55, l=0, r=0),
        font=dict(family=FONT_FAMILY, size=20, color="black"),
        plot_bgcolor="white",
        paper_bgcolor="white",
        barmode="overlay",
        barcornerradius=8,
        legend=dict(
            orientation="h",
            yanchor="bottom",
            y=1.15,
            xanchor="center",
            x=0.5,
        ),
        yaxis=dict(
            range=[0, 1],
            dtick=0.5,
            title=METRIC_LABELS[metric_key],
            showgrid=False,
            zeroline=False,
            ticks="outside",
            showline=True,
            linecolor="#BDBDBD",
            tickcolor="#BDBDBD",
            linewidth=1,
        ),
        xaxis=dict(
            showgrid=False,
            zeroline=False,
            ticks="",
            showline=True,
            linecolor="#BDBDBD",
            linewidth=1,
            tickvals=list(xs),
            ticktext=pathogens,
        ),
    )

    if title:
        fig.update_layout(title=dict(text=title, font=dict(family=FONT_FAMILY, size=20)))

    # if save_path_png:
    #     fig.write_image(save_path_png, scale=1)
    # if save_path_pdf:
    #     fig.write_image(save_path_pdf, scale=1)

    fig.show()
    return fig

plot_metric(metric="recall", save_path_png="../assets/figures/article_screening/article_screening_recall.png", save_path_pdf="../assets/figures/article_screening/article_screening_recall.pdf")

In [18]:
# Average articles passed through each stage.
t_and_a = 9132
full_text = 1102
extraction = 395

df_time_stats = pd.read_csv("../data/agentslr/evals/stats/time_per_article_minutes.csv",)


df_time_stats_oss = df_time_stats[df_time_stats["Model"] == "oss"]
# Creaverage_rowate Overall Average Row (with Pathogen as "Average") and add it to the DataFrame
average_row = df_time_stats_oss.mean(numeric_only=True)

average_row["Pathogen"] = "Average"
average_row["Model"] = "oss"

df_time_stats_oss = pd.concat([df_time_stats_oss, average_row.to_frame().T], ignore_index=True)
avg_oss_per_article = df_time_stats_oss[df_time_stats_oss.Pathogen=='Average']
avg_oss_per_article

,Model,Pathogen,Title & Abstract Screening,Download and Harvest,Full-text Screening,Parameter Extraction,Model Extraction,Outbreak Extraction,PDF-to-MD Conversion
4,oss,Average,0.010429,0.010581,0.033821,1.7195,0.17763,0.137629,0.018333


In [19]:

screening_ai_conditioned_ablation = (avg_oss_per_article['Title & Abstract Screening']* t_and_a) + (avg_oss_per_article['Download and Harvest']* t_and_a) + (avg_oss_per_article['Full-text Screening']* full_text) + (avg_oss_per_article['PDF-to-MD Conversion']* full_text)

screening_direct_ablation = (avg_oss_per_article['Download and Harvest']* t_and_a) + (avg_oss_per_article['Full-text Screening']* t_and_a) + (avg_oss_per_article['PDF-to-MD Conversion']* t_and_a)

print("Time taken to conduct article screening with AI-conditioned ablation (in hours):", round(screening_ai_conditioned_ablation.values[0]/60, 2))
print("Time taken to conduct article screening with direct full-text screening ablation (in hours):", round(screening_direct_ablation.values[0]/60, 2))

print("Cost for Mistral OCR PDF to MD Conversion with AI-conditioned article Screening (in USD):", ((16.6 * full_text)/1000)*2)
print("Cost for Mistral OCR PDF to MD Conversion with Direct Full-text Screening (in USD):", ((16.6 * t_and_a)/1000)*2)

Time taken to conduct article screening with AI-conditioned ablation (in hours): 4.16
Time taken to conduct article screening with direct full-text screening ablation (in hours): 9.55
Cost for Mistral OCR PDF to MD Conversion with AI-conditioned article Screening (in USD): 36.586400000000005
Cost for Mistral OCR PDF to MD Conversion with Direct Full-text Screening (in USD): 303.18240000000003


In [20]:
print("Cost for Mistral OCR PDF to MD Conversion with AI-conditioned article Screening (in USD):", ((16.6 * full_text)/1000)*2)
print("Cost for Mistral OCR PDF to MD Conversion with Direct Full-text Screening (in USD):", ((16.6 * t_and_a)/1000)*2)

Cost for Mistral OCR PDF to MD Conversion with AI-conditioned article Screening (in USD): 36.586400000000005
Cost for Mistral OCR PDF to MD Conversion with Direct Full-text Screening (in USD): 303.18240000000003


## Human Expert Validation

In [21]:
from pathlib import Path
import json
import math
import statistics
from collections import Counter

import plotly.graph_objects as go
from plotly.subplots import make_subplots

DATA_DIR = Path("../data/human_validation")

with open(DATA_DIR / "parameters" / "results.json", "r") as f:
    parameters = json.load(f)

with open(DATA_DIR / "models" / "results.json", "r") as f:
    models = json.load(f)

with open(DATA_DIR / "outbreaks" / "results.json", "r") as f:
    outbreaks = json.load(f)

COLOR_PARAMETERS = "#FFCBA4"
COLOR_MODELS = "#E38B48"
COLOR_OUTBREAKS = "#B45C1A"

FONT_FAMILY = "Times New Roman, Times, serif"
TICK_FONT_SIZE = 21
TITLE_FONT_SIZE = 21

AXIS_LINECOLOR = "#BDBDBD"
AXIS_LINEWIDTH = 1

LABEL_PAD = 0.01

def standard_error(p, n):
    return math.sqrt(p * (1 - p) / n)

categories = ["Parameters", "Models", "Outbreaks"]
data_objects = [parameters, models, outbreaks]
colors = [COLOR_PARAMETERS, COLOR_MODELS, COLOR_OUTBREAKS]

fig = make_subplots(
    rows=3, cols=3,
    subplot_titles=("Expert-Rated Flagging Precision", "Expert-Rated Extraction Accuracy", "Expert-Rated Competence"),
    horizontal_spacing=0.07,
    vertical_spacing=0.06,
    shared_xaxes=True
)

rating_bins = list(range(1, 8))

for row_idx, (cat, obj, color) in enumerate(zip(categories, data_objects, colors), start=1):
    n = len(obj["competence_ratings"])

    precision = obj["flagging"].get("precision", 0)
    precision_se = standard_error(precision, n) if precision else 0

    fig.add_trace(
        go.Bar(
            x=[precision],
            y=[cat],
            orientation="h",
            marker=dict(color=color, line=dict(width=0)),
            name=cat,
            showlegend=True,
            legendgroup=cat,
            error_x=dict(
                type="data",
                array=[precision_se],
                visible=True,
                color="black",
                thickness=0.65,
                width=3
            )
        ),
        row=row_idx, col=1
    )

    fig.add_annotation(
        x=precision + precision_se + LABEL_PAD,
        y=cat,
        text=f"{precision:.2f}",
        showarrow=False,
        font=dict(family=FONT_FAMILY, size=TICK_FONT_SIZE, color="#111111"),
        xanchor="left",
        yanchor="middle",
        row=row_idx, col=1
    )

    accuracy = obj["field_accuracies"]["average"]
    accuracy_se = standard_error(accuracy, n)

    fig.add_trace(
        go.Bar(
            x=[accuracy],
            y=[cat],
            orientation="h",
            marker=dict(color=color, line=dict(width=0)),
            name=cat,
            showlegend=False,
            legendgroup=cat,
            error_x=dict(
                type="data",
                array=[accuracy_se],
                visible=True,
                color="black",
                thickness=0.65,
                width=3
            )
        ),
        row=row_idx, col=2
    )

    fig.add_annotation(
        x=accuracy + accuracy_se + LABEL_PAD,
        y=cat,
        text=f"{accuracy:.2f}",
        showarrow=False,
        font=dict(family=FONT_FAMILY, size=TICK_FONT_SIZE, color="#111111"),
        xanchor="left",
        yanchor="middle",
        row=row_idx, col=2
    )

    ratings = obj["competence_ratings"]
    counts = Counter(ratings)
    total = len(ratings)
    proportions = [counts.get(r, 0) / total for r in rating_bins]

    fig.add_trace(
        go.Bar(
            x=rating_bins,
            y=proportions,
            marker=dict(color=color, line=dict(width=0)),
            name=cat,
            showlegend=False,
            legendgroup=cat
        ),
        row=row_idx, col=3
    )

    mean_rating = statistics.mean(ratings)
    fig.add_vline(
        x=mean_rating,
        row=row_idx, col=3,
        line=dict(color="black", width=1, dash="dash")
    )

bar_tickvals = [0, 0.25, 0.5, 0.75, 1]

for row in range(1, 4):
    is_bottom_row = (row == 3)

    fig.update_xaxes(
        range=[0, 1],
        row=row, col=1,
        dtick=0.5,
        showticklabels=is_bottom_row,
        # tickvals=bar_tickvals if is_bottom_row else None,
        showgrid=False,
        showline=True,
        linecolor=AXIS_LINECOLOR,
        linewidth=AXIS_LINEWIDTH,
        ticks="outside" if is_bottom_row else "",
        tickcolor="#BDBDBD",
        zeroline=False,
        tickfont=dict(family=FONT_FAMILY, size=TICK_FONT_SIZE)
    )
    fig.update_xaxes(
        range=[0, 1],
        row=row, col=2,
        dtick=0.5,
        showticklabels=is_bottom_row,
        # tickvals=bar_tickvals if is_bottom_row else None,
        showgrid=False,
        showline=True,
        linecolor=AXIS_LINECOLOR,
        linewidth=AXIS_LINEWIDTH,
        ticks="outside" if is_bottom_row else "",
        tickcolor="#BDBDBD",
        zeroline=False,
        tickfont=dict(family=FONT_FAMILY, size=TICK_FONT_SIZE)
    )
    fig.update_xaxes(
        range=[0.5, 7.5],
        row=row, col=3,
        showticklabels=is_bottom_row,
        tickvals=[1, 2, 3, 4, 5, 6, 7] if is_bottom_row else None,
        ticktext=["1<br>Incompetent", "2", "3", "4", "5", "6", "7<br>Autonomous"] if is_bottom_row else None,
        tickangle=0,
        showgrid=False,
        showline=True,
        linecolor=AXIS_LINECOLOR,
        linewidth=AXIS_LINEWIDTH,
        ticks="outside" if is_bottom_row else "",
        tickcolor="#BDBDBD",
        zeroline=False,
        tickfont=dict(family=FONT_FAMILY, size=TICK_FONT_SIZE)
    )

for row in range(1, 4):
    fig.update_yaxes(
        showticklabels=True,
        row=row, col=1,
        tickfont=dict(family=FONT_FAMILY, size=TICK_FONT_SIZE),
        showline=True,
        linecolor=AXIS_LINECOLOR,
        linewidth=AXIS_LINEWIDTH,
        ticks="",
        mirror=False,
        zeroline=False,
        showgrid=False
    )
    fig.update_yaxes(
        showticklabels=False,
        row=row, col=2,
        showline=True,
        linecolor=AXIS_LINECOLOR,
        linewidth=AXIS_LINEWIDTH,
        ticks="",
        mirror=False,
        zeroline=False,
        showgrid=False
    )
    fig.update_yaxes(
        showticklabels=True,
        row=row, col=3,
        ticks="",
        dtick=0.2,
        showgrid=False,
        tickfont=dict(family=FONT_FAMILY, size=TICK_FONT_SIZE),
        showline=True,
        linecolor=AXIS_LINECOLOR,
        linewidth=AXIS_LINEWIDTH,
        mirror=False,
        zeroline=False
    )

fig.update_layout(
    height=450,
    width=1200,
    font=dict(family=FONT_FAMILY, color="black", size=TICK_FONT_SIZE),
    plot_bgcolor="white",
    paper_bgcolor="white",
    margin=dict(t=60, b=70, l=80, r=40),
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.12,
        xanchor="center",
        x=0.5,
        font=dict(family=FONT_FAMILY, size=TICK_FONT_SIZE),
        bgcolor="rgba(255,255,255,0)",
        borderwidth=0,
        entrywidth=170,
        entrywidthmode="pixels",
    ),
    barcornerradius=8
)

for annotation in fig["layout"]["annotations"]:
    if annotation.xref == "paper":
        annotation.update(
            font=dict(family=FONT_FAMILY, color="black", size=TITLE_FONT_SIZE),
            yshift=5
        )

fig.show()

In [22]:
import math

names = ["Parameters", "Models", "Outbreaks"]

group_orders = {
    "Parameters": ["value", "uncertainty", "population", "aggregation"],
    "Outbreaks": ["temporal", "geographical", "case_burden", "epidemiological"]
}

section_labels = {
    "value": "Value fields",
    "uncertainty": "Uncertainty fields",
    "population": "Population fields",
    "aggregation": "Aggregation fields",
    "temporal": "Temporal fields",
    "geographical": "Geographical fields",
    "case_burden": "Case burden fields",
    "epidemiological": "Epidemiological fields"
}

group_labels = {
    "value": "Value",
    "uncertainty": "Uncertainty",
    "population": "Population",
    "aggregation": "Aggregation",
    "temporal": "Temporal",
    "geographical": "Geographical",
    "case_burden": "Case burden",
    "epidemiological": "Epidemiological"
}

field_labels = {
    "attack_rate": "Attack rate",
    "growth_rate": "Growth rate",
    "human_delay": "Human delay",
    "reproduction_number": "Reproduction number",
    "seroprevalence": "Seroprevalence",
    "severity": "Severity",
    "value": "Value",
    "unit": "Unit",
    "type": "Type",
    "bounds": "Bounds",
    "value_type": "Value type",
    "statistical_approach": "Statistical approach",
    "single_type_uncertainty": "Single-type uncertainty",
    "paired_uncertainty": "Paired uncertainty",
    "distribution_type": "Distribution type",
    "population_sample_type": "Sample type",
    "population_group": "Population group",
    "population_sample_size": "Sample size",
    "population_sex": "Sex",
    "population_age_range": "Age range",
    "population_countries": "Countries",
    "population_locations": "Locations",
    "method_moment_value": "Method moment value",
    "aggregation": "Aggregation",
    "model_type": "Model type",
    "compartmental_type": "Compartmental type",
    "stoch_deter": "Stochastic or deterministic",
    "theoretical_model": "Theoretical model",
    "outbreak_start_year": "Start year",
    "outbreak_start_month": "Start month",
    "outbreak_start_day": "Start day",
    "outbreak_end_year": "End year",
    "outbreak_end_month": "End month",
    "outbreak_end_day": "End day",
    "outbreak_duration_months": "Duration in months",
    "outbreak_country": "Country",
    "outbreak_location": "Location",
    "cases_confirmed": "Confirmed cases",
    "cases_suspected": "Suspected cases",
    "cases_asymptomatic": "Asymptomatic cases",
    "cases_severe": "Severe cases",
    "deaths": "Deaths",
    "mode_of_detection": "Mode of detection",
    "pre_outbreak": "Pre-outbreak status",
    "asymptomatic_transmission_described": "Asymptomatic transmission described"
}

parameter_precision_by_class = {
    "attack_rate": 0.25,
    "growth_rate": 1.0,
    "human_delay": 0.625,
    "reproduction_number": 1.0,
    "seroprevalence": 0.5,
    "severity": 0.5714285714285714
}

parameter_precision_order = [
    "attack_rate",
    "growth_rate",
    "human_delay",
    "reproduction_number",
    "seroprevalence",
    "severity"
]

model_field_order = [
    "model_type",
    "compartmental_type",
    "stoch_deter",
    "theoretical_model"
]

field_orders = {
    "value": [
        "value",
        "unit",
        "type",
        "bounds",
        "value_type",
        "statistical_approach"
    ],
    "uncertainty": [
        "single_type_uncertainty",
        "paired_uncertainty",
        "distribution_type"
    ],
    "population": [
        "population_sample_type",
        "population_group",
        "population_sample_size",
        "population_sex",
        "population_age_range",
        "population_countries",
        "population_locations",
        "method_moment_value"
    ],
    "aggregation": [
        "aggregation"
    ],
    "temporal": [
        "outbreak_start_year",
        "outbreak_start_month",
        "outbreak_start_day",
        "outbreak_end_year",
        "outbreak_end_month",
        "outbreak_end_day",
        "outbreak_duration_months"
    ],
    "geographical": [
        "outbreak_country",
        "outbreak_location"
    ],
    "case_burden": [
        "cases_confirmed",
        "cases_suspected",
        "cases_asymptomatic",
        "cases_severe",
        "deaths"
    ],
    "epidemiological": [
        "mode_of_detection",
        "pre_outbreak",
        "asymptomatic_transmission_described"
    ]
}

def fmt(v):
    if v is None:
        return ""
    if isinstance(v, float) and math.isnan(v):
        return ""
    return f"{v:.2f}"

lines = []
lines.append(r"\begin{footnotesize}")
lines.append(r"\begin{longtable}{>{\raggedright\arraybackslash}p{10.8cm} S[table-format=1.2]}")
lines.append(r"\caption{\textbf{Extended expert validation results.} Results are reported as expert-rated flagging precision and expert-rated extraction accuracy. Within each section, rows are ordered from overall scores to subgroup scores and then field-level scores.}")
lines.append(r"\label{tab:extended_expert_validation_longtable} \\")
lines.append(r"\toprule")
lines.append(r"\textbf{Item} & {\textbf{Score}} \\")
lines.append(r"\midrule")
lines.append(r"\endfirsthead")
lines.append("")
lines.append(r"\multicolumn{2}{l}{\textit{Table \thetable\ continued from previous page}} \\")
lines.append(r"\toprule")
lines.append(r"\textbf{Item} & {\textbf{Score}} \\")
lines.append(r"\midrule")
lines.append(r"\endhead")
lines.append("")
lines.append(r"\midrule")
lines.append(r"\multicolumn{2}{r}{\textit{Continued on next page}} \\")
lines.append(r"\endfoot")
lines.append("")
lines.append(r"\bottomrule")
lines.append(r"\endlastfoot")
lines.append("")

for name, obj in zip(names, data_objects):
    lines.append(rf"\multicolumn{{2}}{{l}}{{\textbf{{{name}}}}} \\")
    lines.append(r"\addlinespace[0.2em]")
    lines.append(rf"Overall --- Flagging precision & {fmt(obj['flagging']['precision'])} \\")
    lines.append(rf"Overall --- Extraction accuracy & {fmt(obj['field_accuracies']['average'])} \\")
    lines.append("")

    if name == "Parameters":
        lines.append(r"\addlinespace[0.35em]")
        lines.append(r"\textit{Precision by class} & {} \\")
        for key in parameter_precision_order:
            lines.append(rf"\quad {field_labels.get(key, key)} & {fmt(parameter_precision_by_class[key])} \\")
        lines.append("")

    groups = obj["field_accuracies"].get("groups", {})
    fields = obj["field_accuracies"].get("fields", {})

    if name in group_orders:
        lines.append(r"\addlinespace[0.35em]")
        lines.append(r"\textit{Accuracy by group} & {} \\")
        for group_key in group_orders[name]:
            if group_key in groups:
                lines.append(rf"\quad {group_labels[group_key]} & {fmt(groups[group_key])} \\")
        lines.append("")

    if name == "Models":
        lines.append(r"\addlinespace[0.35em]")
        lines.append(r"\textit{Field accuracy} & {} \\")
        for field_key in model_field_order:
            value = fields.get(field_key)
            if value is not None and not (isinstance(value, float) and math.isnan(value)):
                lines.append(rf"\quad {field_labels.get(field_key, field_key)} & {fmt(value)} \\")
        lines.append("")
    else:
        for group_key in group_orders.get(name, []):
            ordered_field_keys = field_orders.get(group_key, [])
            group_rows = []
            for field_key in ordered_field_keys:
                meta = fields.get(field_key)
                if isinstance(meta, dict):
                    acc = meta.get("accuracy")
                    if acc is not None and not (isinstance(acc, float) and math.isnan(acc)):
                        group_rows.append((field_key, acc))
            if group_rows:
                lines.append(r"\addlinespace[0.35em]")
                lines.append(rf"\textit{{{section_labels[group_key]}}} & {{}} \\")
                for field_key, acc in group_rows:
                    lines.append(rf"\quad {field_labels.get(field_key, field_key)} & {fmt(acc)} \\")
                lines.append("")

    lines.append(r"\addlinespace[0.6em]")

if lines[-1] == r"\addlinespace[0.6em]":
    lines.pop()

lines.append(r"\end{longtable}")
lines.append(r"\end{footnotesize}")

latex_table = "\n".join(lines)
print(latex_table)

\begin{footnotesize}
\begin{longtable}{>{\raggedright\arraybackslash}p{10.8cm} S[table-format=1.2]}
\caption{\textbf{Extended expert validation results.} Results are reported as expert-rated flagging precision and expert-rated extraction accuracy. Within each section, rows are ordered from overall scores to subgroup scores and then field-level scores.}
\label{tab:extended_expert_validation_longtable} \\
\toprule
\textbf{Item} & {\textbf{Score}} \\
\midrule
\endfirsthead

\multicolumn{2}{l}{\textit{Table \thetable\ continued from previous page}} \\
\toprule
\textbf{Item} & {\textbf{Score}} \\
\midrule
\endhead

\midrule
\multicolumn{2}{r}{\textit{Continued on next page}} \\
\endfoot

\bottomrule
\endlastfoot

\multicolumn{2}{l}{\textbf{Parameters}} \\
\addlinespace[0.2em]
Overall --- Flagging precision & 0.66 \\
Overall --- Extraction accuracy & 0.77 \\

\addlinespace[0.35em]
\textit{Precision by class} & {} \\
\quad Attack rate & 0.25 \\
\quad Growth rate & 1.00 \\
\quad Human delay & 

## Model Ablations — Clients

In [23]:
import math
import base64
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from pathlib import Path

try:
    import cairosvg
except Exception:
    cairosvg = None

METRIC = "F1"  # Options: "F1", "Precision", "Recall"

FONT = "Times New Roman, Times, serif"
FONT_SIZE = 22
BG_COLOR = "white"

COLOR_TITLE_ABSTRACT = "#B7D3C6"
COLOR_FULLTEXT_SCREENING = "#6BAED6"
COLOR_PARAMETERS = "#FFCBA4"
COLOR_MODELS = "#E38B48"
COLOR_OUTBREAKS = "#B45C1A"
HEADER_ALPHA = 0.30

MODEL_NAME_MAP = {
    "oss": "GPT-OSS-120B (High)",
    "gpt": "GPT-5.2 (High)",
    "deepseek": "DeepSeek-V3.2",
    "kimi": "Kimi-K2.5",
    "glm": "GLM-4.7",
}
MODEL_CSV_ORDER = ["oss", "glm", "deepseek", "kimi", "gpt",]
MODELS = [MODEL_NAME_MAP[m] for m in MODEL_CSV_ORDER]


COMP_COLORS = {
    "Flagging": "#D97C7C",
    "Counts": "#8BB3E6",
    "Extraction": "#8FD3B0",
}
UNIFORM_DASH = "6px,6px"
LINE_ALPHA = 0.70
LINE_WIDTH = 2.2

ERR_COLOR = "#000000" 
ERR_THICKNESS = 1.5
ERR_CAP = 5

LABEL_PAD = 0.06
LABEL_PAD_BEST_EXTRA = 0.001

LOGO_PX = 80
LOGO_OFFSET_PX = 14
LOGO_SIZEY = 0.20
LOGO_SIZEX = 0.60

DOT_SIZE = 9
DOT_X_OFFSET = -0.45
DOT_LINE_W = 1.4

BAR_CORNER_RADIUS = 8

DATA_DIR = Path("../data/agentslr/evals")
ICON_DIR = Path("../assets/icons")


In [24]:
def hex_to_rgba(hex_color, alpha):
    h = hex_color.strip().lstrip("#")
    r, g, b = int(h[0:2], 16), int(h[2:4], 16), int(h[4:6], 16)
    return f"rgba({r},{g},{b},{alpha})"


def svg_path_to_png_data_uri(svg_path, out_px=72):
    svg_bytes = Path(svg_path).read_bytes()
    if cairosvg is None:
        encoded = base64.b64encode(svg_bytes).decode("ascii")
        return f"data:image/svg+xml;base64,{encoded}"
    png_bytes = cairosvg.svg2png(bytestring=svg_bytes, output_width=out_px, output_height=out_px)
    encoded = base64.b64encode(png_bytes).decode("ascii")
    return f"data:image/png;base64,{encoded}"


def model_to_icon_key(model_name):
    s = model_name.lower()
    if "deepseek" in s:
        return "deepseek"
    if "kimi" in s:
        return "kimi"
    if "glm" in s:
        return "zai"
    return "openai"


ICON_SOURCES = {
    "deepseek": svg_path_to_png_data_uri(str(ICON_DIR / "deepseek.svg"), out_px=LOGO_PX),
    "kimi": svg_path_to_png_data_uri(str(ICON_DIR / "kimi.svg"), out_px=LOGO_PX),
    "openai": svg_path_to_png_data_uri(str(ICON_DIR / "openai.svg"), out_px=LOGO_PX),
    "zai": svg_path_to_png_data_uri(str(ICON_DIR / "zai.svg"), out_px=LOGO_PX),
}


def load_screening_summary(filepath):
    df = pd.read_csv(filepath, header=[0, 1])
    new_cols = []
    for col in df.columns:
        level0, level1 = col[0].strip(), col[1].strip()
        if "Unnamed" in level1 or level1 == "":
            new_cols.append("pathogen")
        else:
            new_cols.append(f"{level0}_{level1}")
    df.columns = new_cols
    return df


def compute_screening_stats(df, model_csv_names, metric):
    metric_key = metric.lower()
    df_pathogens = df[df["pathogen"] != "Overall"].copy()
    results = {}
    for model in model_csv_names:
        col = f"{model}_{metric_key}"
        if col in df_pathogens.columns:
            values = df_pathogens[col].astype(float)
            results[model] = {"mean": values.mean(), "std": values.std()}
    return results


def get_extraction_stats_from_aggregated(filepath, model_csv_names, metric):
    df = pd.read_csv(filepath)
    df_metric = df[df["metric"] == metric].copy()
    results = {}
    for model in model_csv_names:
        df_model = df_metric[df_metric["model"] == model]
        if len(df_model) == 0:
            results[model] = {
                "mean": 0.0,
                "std": 0.0,
                "components": {"flagging": 0.0, "count": 0.0, "extraction": 0.0},
            }
            continue
        per_pathogen = df_model.groupby("pathogen")[["flagging_score", "count_score", "extraction_score", "overall_score"]].mean()
        results[model] = {
            "mean": per_pathogen["overall_score"].mean(),
            "std": per_pathogen["overall_score"].std(),
            "components": {
                "flagging": per_pathogen["flagging_score"].mean(),
                "count": per_pathogen["count_score"].mean(),
                "extraction": per_pathogen["extraction_score"].mean(),
            },
        }
    return results

In [26]:

abstract_df = load_screening_summary(DATA_DIR / "article_screening" / "abstract_screening_summary.csv")
# fulltext_df = load_screening_summary(DATA_DIR / "article_screening" / "fulltext_screening_summary.csv")
fulltext_df = load_screening_summary(DATA_DIR / "article_screening" / "fulltext_screening_summary_perg_conditioned.csv")

abstract_stats = compute_screening_stats(abstract_df, MODEL_CSV_ORDER, METRIC)
fulltext_stats = compute_screening_stats(fulltext_df, MODEL_CSV_ORDER, METRIC)

params_stats = get_extraction_stats_from_aggregated(
    DATA_DIR / "data_extraction" / "parameters" / "extraction_parameters_aggregated.csv",
    MODEL_CSV_ORDER, METRIC,
)
models_stats = get_extraction_stats_from_aggregated(
    DATA_DIR / "data_extraction" / "models" / "extraction_models_aggregated.csv",
    MODEL_CSV_ORDER, METRIC,
)
outbreaks_stats = get_extraction_stats_from_aggregated(
    DATA_DIR / "data_extraction" / "outbreaks" / "extraction_outbreaks_aggregated.csv",
    MODEL_CSV_ORDER, METRIC,
)

STAGES_META = [
    ("Title & Abstract Screening", abstract_stats, False, COLOR_TITLE_ABSTRACT),
    ("Full-text Screening", fulltext_stats, False, COLOR_FULLTEXT_SCREENING),
    ("Parameter Extraction", params_stats, True, COLOR_PARAMETERS),
    ("Model Extraction", models_stats, True, COLOR_MODELS),
    ("Outbreak Extraction", outbreaks_stats, True, COLOR_OUTBREAKS),
]

master_rows = []
for stage_name, stats, has_components, _ in STAGES_META:
    for model_csv in MODEL_CSV_ORDER:
        model_display = MODEL_NAME_MAP[model_csv]
        if model_csv not in stats:
            continue
        s = stats[model_csv]
        row = {
            "stage": stage_name,
            "model_csv": model_csv,
            "model": model_display,
            "f1_mean": s["mean"],
            "f1_std": s["std"],
            "has_components": has_components,
            "flagging": s["components"]["flagging"] if has_components and "components" in s else None,
            "counts": s["components"]["count"] if has_components and "components" in s else None,
            "extraction": s["components"]["extraction"] if has_components and "components" in s else None,
        }
        master_rows.append(row)

master_df = pd.DataFrame(master_rows)

In [27]:
def build_fig(MODEL_COLORS, COMP_COLORS, STAGES_META):
    fig = make_subplots(rows=1, cols=len(STAGES_META), horizontal_spacing=0.02)

    MODEL_LEGEND_NAME_MAP = {
        "GPT-OSS-120B (High)": "GPT-OSS-120B",
        "GPT-5.2 (High)": "GPT-5.2",
        "DeepSeek-V3.2": "DeepSeek-V3.2",
        "Kimi-K2.5": "Kimi-K2.5",
        "GLM-4.7": "GLM-4.7",
    }

    for m in MODELS:
        # fig.add_trace(
        #     go.Scatter(
        #         x=[None], y=[None], mode="markers",
        #         marker=dict(symbol="square", size=14, color=MODEL_COLORS[m], cornerradius=BAR_CORNER_RADIUS, ),
        #         name=MODEL_LEGEND_NAME_MAP.get(m, m), showlegend=True, legendgroup="models", hoverinfo="skip",
        #     ),
        #     row=1, col=1,
        # )
        fig.add_trace(
        go.Bar(
            x=[0], y=[0],  # dummy
            marker=dict(
                color=MODEL_COLORS[m],
                line=dict(width=0),
                cornerradius=BAR_CORNER_RADIUS,
            ),
            name=MODEL_LEGEND_NAME_MAP.get(m, m),
            showlegend=True,
            legendgroup="models",
            hoverinfo="skip",
            # visible="legendonly",
            # opacity=1
        ),
        row=1, col=1,
    )

    

    fig.add_trace(
        go.Scatter(
            x=[None], y=[None], mode="markers",
            marker=dict(size=0, opacity=0),
            name="\u00A0" * 12, showlegend=True, hoverinfo="skip", legendgroup="spacer",
        ),
        row=1, col=1,
    )

    for comp in ["Flagging", "Counts", "Extraction"]:
        fig.add_trace(
            go.Scatter(
                x=[None], y=[None], mode="markers",
                marker=dict(symbol="circle", size=14, color=COMP_COLORS[comp]),
                name=comp, showlegend=True, legendgroup="components", hoverinfo="skip",
            ),
            row=1, col=1,
        )

    ymin, ymax = 0, 1
    layout_h = 450
    layout_margin_t = 20
    layout_margin_b = 5
    plot_h_px = max(1, layout_h - layout_margin_t - layout_margin_b)
    px_to_data = (ymax - ymin) / plot_h_px
    logo_offset_data = LOGO_OFFSET_PX * px_to_data

    ylabel = {"F1": "F<sub>1</sub> Score", "Precision": "Macro Precision", "Recall": "Macro Recall"}.get(METRIC, f"Macro {METRIC}")

    for col_idx, (stage_name, _, has_comp, _) in enumerate(STAGES_META, start=1):
        stage_data = master_df[master_df["stage"] == stage_name].copy()
        vals, errs, components_data = [], [], {}

        for model_display in MODELS:
            row = stage_data[stage_data["model"] == model_display]
            if len(row) > 0:
                v = float(row["f1_mean"].values[0])
                e = float(row["f1_std"].values[0])
                vals.append(v)
                errs.append(e)
                if bool(row["has_components"].values[0]):
                    components_data[model_display] = {
                        "Flagging": float(row["flagging"].values[0]),
                        "Counts": float(row["counts"].values[0]),
                        "Extraction": float(row["extraction"].values[0]),
                    }
            else:
                vals.append(0.0)
                errs.append(0.0)

        best_j = max(range(len(MODELS)), key=lambda j: vals[j]) if len(MODELS) else None

        for rank, orig_j in enumerate(range(len(MODELS))):
            model_name = MODELS[orig_j]
            v, e = vals[orig_j], errs[orig_j]
            is_best = (orig_j == best_j)

            fig.add_trace(
                go.Bar(
                    x=[rank], y=[v], width=0.9,
                    marker=dict(
                        color=MODEL_COLORS[model_name],
                        line=dict(width=0),
                        cornerradius=BAR_CORNER_RADIUS,
                    ),
                    error_y=dict(
                        type="data", array=[e], symmetric=True,
                        thickness=ERR_THICKNESS, width=ERR_CAP, color=ERR_COLOR,
                    ),
                    showlegend=False, hoverinfo="skip",
                ),
                row=1, col=col_idx,
            )

            ax_suffix = "" if col_idx == 1 else str(col_idx)
            xref, yref = f"x{ax_suffix}", f"y{ax_suffix}"
            label_y = (v + e) + LABEL_PAD + (LABEL_PAD_BEST_EXTRA if is_best else 0)

            fig.add_annotation(
                xref=xref, yref=yref, x=rank, y=label_y,
                text=f"{v:.2f}" if not is_best else f"<b>{v:.2f}</b>",
                showarrow=False,
                font=dict(family=FONT, size=FONT_SIZE, color="#111111" if is_best else "#555555"),
            )

            # icon_src = ICON_SOURCES[model_to_icon_key(model_name)]
            # sizey = min(LOGO_SIZEY, max(0.0, v - 0.01))
            # y_center = (v - e) - logo_offset_data - sizey / 2
            # if sizey > 0:
            #     fig.add_layout_image(dict(
            #         source=icon_src, xref=xref, yref=yref,
            #         x=rank, y=0.15,
            #         xanchor="center", yanchor="middle",
            #         sizex=LOGO_SIZEX, sizey=sizey,
            #         sizing="contain", layer="above",
            #     ))

            if model_name in components_data:
                for comp_name, cv in components_data[model_name].items():
                    if cv is None or cv <= 0:
                        continue
                    fig.add_trace(
                        go.Scatter(
                            x=[rank + DOT_X_OFFSET], y=[cv], mode="markers",
                            marker=dict(
                                symbol="circle", size=DOT_SIZE,
                                color=COMP_COLORS[comp_name],
                                line=dict(color=COMP_COLORS[comp_name], width=DOT_LINE_W),
                            ),
                            showlegend=False, hoverinfo="skip",
                        ),
                        row=1, col=col_idx,
                    )

    xmax = (len(MODELS) - 1) + 0.55

    for col in range(1, len(STAGES_META) + 1):
        show_yticks = col == 1
        fig.update_yaxes(
            range=[ymin, ymax], domain=[0.15, 1.0],
            dtick=0.5,
            showgrid=False, zeroline=False,
            showticklabels=show_yticks,
            ticks="outside" if show_yticks else "",
            tickcolor="#BDBDBD",
            tickfont=dict(family=FONT, size=FONT_SIZE, color="#111111"),
            showline=show_yticks,
            linecolor="#BDBDBD", linewidth=1,
            title_text=ylabel if show_yticks else None,
            title=dict(
                text=ylabel if show_yticks else "",
                font=dict(family=FONT, size=24, color="#111111"),
                standoff=20,
            ),
            row=1, col=col,
        )
        fig.update_xaxes(
            showgrid=False, zeroline=False, showticklabels=False,
            showline=True, linecolor="#BDBDBD", linewidth=1,
            range=[-0.55, xmax], row=1, col=col,
        )

    shapes, annotations = [], []
    spacing = 0.007
    n_cols = len(STAGES_META)
    panel_w = (1.0 - spacing * (n_cols - 1)) / n_cols

    for i, (header_text, _, _, header_color) in enumerate(STAGES_META):
        x0 = i * (panel_w + spacing)
        x1 = x0 + panel_w
        shapes.append(dict(
            type="rect", xref="paper", yref="paper",
            x0=x0, x1=x1, y0=0.0, y1=0.1,
            # fillcolor=hex_to_rgba(header_color, HEADER_ALPHA),
            line=dict(color="rgba(0,0,0,0)", width=0), layer="above",
        ))
        annotations.append(dict(
            xref="paper", yref="paper",
            x=(x0 + x1) / 2, y=0.095,
            text=header_text, showarrow=False,
            xanchor="center", yanchor="middle",
            font=dict(family=FONT, size=24, color="#111111"),
            align="center",
        ))

    fig.update_layout(
        height=layout_h, width=1440,
        font=dict(family=FONT, size=FONT_SIZE, color="#111111"),
        plot_bgcolor=BG_COLOR, paper_bgcolor=BG_COLOR,
        margin=dict(t=0, b=0, l=0, r=0),
        barmode="overlay", bargap=0.18,
        barcornerradius=BAR_CORNER_RADIUS,
        shapes=shapes,
        annotations=fig.layout.annotations + tuple(annotations),
        legend=dict(
            orientation="h", yanchor="bottom", y=1.04, xanchor="center", x=0.5,
            font=dict(family=FONT, size=24, color="#111111"),
            bgcolor="rgba(255,255,255,0)", borderwidth=0, traceorder="normal",
        ),
    )

    return fig

MODEL_COLORS = {
    "GPT-OSS-120B (High)": "#2B2E5F",
    "GLM-4.7":             "#4B4F86",
    "DeepSeek-V3.2":       "#6C74AD",
    "Kimi-K2.5":           "#9AA4CF",
    "GPT-5.2 (High)":      "#BDBDBD"
}


fig = build_fig(MODEL_COLORS, COMP_COLORS, STAGES_META)
# fig.write_image("../assets/figures/model_ablations/model_ablations.pdf",scale=1)
# fig.write_image("../assets/figures/model_ablations/model_ablations.svg",scale=1)
fig.show()

### Appendix Table for Model Ablations

In [28]:
ABSTRACT_CSV = Path("../data/agentslr/evals/article_screening/abstract_screening_summary.csv")
FULLTEXT_CSV = Path("../data/agentslr/evals/article_screening/fulltext_screening_summary.csv")
PARAMS_CSV = Path("../data/agentslr/evals/data_extraction/parameters/extraction_parameters_aggregated.csv")
MODELS_CSV = Path("../data/agentslr/evals/data_extraction/models/extraction_models_aggregated.csv")
OUTBREAKS_CSV = Path("../data/agentslr/evals/data_extraction/outbreaks/extraction_outbreaks_aggregated.csv")

MODEL_ORDER = ["oss", "gpt", "deepseek", "kimi", "glm"]
MODEL_NAME_MAP_LATEX = {
    "oss": "GPT-OSS-120B",
    "gpt": "GPT-5.2",
    "deepseek": "DeepSeek-V3.2",
    "kimi": "Kimi-K2.5",
    "glm": "GLM-4.7",
}
METRIC_MAP = {"precision": "P", "recall": "R", "f1": "F1", "Precision": "P", "Recall": "R", "F1": "F1"}
METRIC_ORDER = ["P", "R", "F1"]

PATHOGENS_BY_STAGE = {
    "Title and abstract screening": ["Marburg", "Ebola", "Lassa", "SARS", "Zika", "MERS", "Nipah", "Overall"],
    "Full-text screening": ["Marburg", "Ebola", "Lassa", "SARS", "Zika", "MERS", "Nipah", "Overall"],
    "Parameter extraction": ["Ebola", "Lassa", "SARS", "Zika", "Overall"],
    "Model extraction": ["Ebola", "Lassa", "SARS", "Zika", "Overall"],
    "Outbreak extraction": ["Lassa", "Zika", "Overall"],
}
KIND_ORDER = ["Flagging", "Counts", "Extraction", "Average"]

In [33]:
def _fmt(x):
    if pd.isna(x):
        return ""
    try:
        return f"{float(x):.2f}"
    except Exception:
        return ""

def _wrap(cell, wraps):
    out = cell
    for w in wraps:
        out = f"{w}{{{out}}}"
    return out

def _read_screening_wide(path):
    df = pd.read_csv(path, header=[0, 1])
    c0 = [("" if (x is None or (isinstance(x, float) and np.isnan(x))) else str(x)).strip() for x in df.columns.get_level_values(0)]
    c1 = [("" if (x is None or (isinstance(x, float) and np.isnan(x))) else str(x)).strip() for x in df.columns.get_level_values(1)]
    df.columns = pd.MultiIndex.from_arrays([c0, c1])
    pathogen_col = None
    for col in df.columns:
        if str(col[0]).strip().lower() == "pathogen":
            pathogen_col = col
            break
    if pathogen_col is None:
        raise ValueError(f"Could not find 'pathogen' column in {path}")
    out = pd.DataFrame()
    out["Pathogen"] = df[pathogen_col].astype(str).str.strip()
    for m in MODEL_ORDER:
        for met in ["precision", "recall", "f1"]:
            cand = None
            for col in df.columns:
                if str(col[0]).strip().lower() == m and str(col[1]).strip().lower() == met:
                    cand = col
                    break
            if cand is not None:
                out[(m, METRIC_MAP[met])] = pd.to_numeric(df[cand], errors="coerce")
    return out

def _read_extraction(path):
    df = pd.read_csv(path)
    df.columns = [str(c).strip() for c in df.columns]
    req = {"model", "pathogen", "metric", "flagging_score", "count_score", "extraction_score"}
    if not req.issubset(set(df.columns)):
        raise ValueError(f"Missing columns in {path}: {sorted(list(req - set(df.columns)))}")
    df["model"] = df["model"].astype(str).str.strip().str.lower()
    df["pathogen"] = df["pathogen"].astype(str).str.strip()
    df["metric"] = df["metric"].astype(str).str.strip()
    df = df[df["model"].isin(MODEL_ORDER)].copy()
    for c in ["flagging_score", "count_score", "extraction_score"]:
        df[c] = pd.to_numeric(df[c], errors="coerce")
    df["overall_score_recalc"] = df[["flagging_score", "count_score", "extraction_score"]].mean(axis=1)
    long = df.melt(
        id_vars=["model", "pathogen", "metric"],
        value_vars=["flagging_score", "count_score", "extraction_score", "overall_score_recalc"],
        var_name="kind",
        value_name="score",
    )
    kind_map = {
        "flagging_score": "Flagging",
        "count_score": "Counts",
        "extraction_score": "Extraction",
        "overall_score_recalc": "Average",
    }
    long["Kind"] = long["kind"].map(kind_map)
    long["Metric"] = long["metric"].map(METRIC_MAP).fillna(long["metric"])
    long = long.drop(columns=["kind"]).copy()
    return long

def _make_extraction_table_df(long, pathogens_keep):
    long = long.copy()
    long = long[long["pathogen"].isin([p for p in pathogens_keep if p != "Overall"])].copy()
    piv = long.pivot_table(
        index=["pathogen", "Kind"],
        columns=["model", "Metric"],
        values="score",
        aggfunc="first",
    )
    rows = []
    for (p, k) in piv.index:
        rec = {"Pathogen": p, "Kind": k}
        for m in MODEL_ORDER:
            for met in METRIC_ORDER:
                if (m, met) in piv.columns:
                    rec[(m, met)] = piv.loc[(p, k), (m, met)]
        rows.append(rec)
    df = pd.DataFrame(rows)
    overall_rows = []
    for k in KIND_ORDER:
        sub = df[df["Kind"] == k].copy()
        if len(sub) == 0:
            continue
        rec = {"Pathogen": "Overall", "Kind": k}
        for m in MODEL_ORDER:
            for met in METRIC_ORDER:
                col = (m, met)
                if col in df.columns:
                    rec[col] = pd.to_numeric(sub[col], errors="coerce").mean()
        overall_rows.append(rec)
    if overall_rows:
        df = pd.concat([df, pd.DataFrame(overall_rows)], ignore_index=True)
    p_order = [p for p in pathogens_keep if p != "Overall"]
    df["Pathogen"] = df["Pathogen"].astype(str).str.strip()
    df["Kind"] = df["Kind"].astype(str).str.strip()
    p_cat = pd.Categorical(df["Pathogen"], categories=p_order + ["Overall"], ordered=True)
    k_cat = pd.Categorical(df["Kind"], categories=KIND_ORDER, ordered=True)
    df = df.assign(_p=p_cat, _k=k_cat).sort_values(["_p", "_k"]).drop(columns=["_p", "_k"]).reset_index(drop=True)
    return df

def _high_low_maps_screening(df):
    models = [m for m in MODEL_ORDER if (m, "F1") in df.columns]
    high = set()
    low = set()
    sub = df[~df["Pathogen"].astype(str).str.strip().eq("Overall")].copy()
    for m in models:
        vals = pd.to_numeric(sub[(m, "F1")], errors="coerce")
        if vals.notna().sum() == 0:
            continue
        mx = vals.max()
        mn = vals.min()
        for idx in sub.index[vals.eq(mx)]:
            high.add((idx, m))
        for idx in sub.index[vals.eq(mn)]:
            low.add((idx, m))
    return high, low, models

def _best_second_model_map(df, by_kind):
    models = [m for m in MODEL_ORDER if (m, "F1") in df.columns]
    best = set()
    second = set()
    for i, row in df.iterrows():
        vals = {m: pd.to_numeric(row.get((m, "F1"), np.nan), errors="coerce") for m in models}
        arr = np.array([v for v in vals.values() if not pd.isna(v)], dtype=float)
        if arr.size == 0:
            continue
        mx = float(np.nanmax(arr))
        best_models = [m for m, v in vals.items() if not pd.isna(v) and float(v) == mx]
        for m in best_models:
            best.add((i, m))
        lower = np.array([float(v) for v in vals.values() if not pd.isna(v) and float(v) < mx], dtype=float)
        if lower.size == 0:
            continue
        sx = float(np.nanmax(lower))
        for m, v in vals.items():
            if not pd.isna(v) and float(v) == sx:
                second.add((i, m))
    return best, second, models

def _latex_screening(df, stage_title, label, caption):
    df = df.copy()
    keep = PATHOGENS_BY_STAGE[stage_title]
    df = df[df["Pathogen"].isin(keep)].copy()
    p_cat = pd.Categorical(df["Pathogen"], categories=keep, ordered=True)
    df = df.assign(_p=p_cat).sort_values("_p").drop(columns=["_p"]).reset_index(drop=True)
    models_present = [m for m in MODEL_ORDER if any((m, met) in df.columns for met in METRIC_ORDER)]
    high, low, _ = _high_low_maps_screening(df)
    best, second, _ = _best_second_model_map(df, by_kind=False)
    colspec = "l|" + "|".join(["ccc"] * len(models_present))
    header1 = "\\textbf{Pathogen} & " + " & ".join(
        [f"\\multicolumn{{3}}{{c|}}{{\\textbf{{{MODEL_NAME_MAP_LATEX[m]}}}}}" for m in models_present[:-1]] +
        ([f"\\multicolumn{{3}}{{c}}{{\\textbf{{{MODEL_NAME_MAP_LATEX[models_present[-1]]}}}}}"] if models_present else [])
    ) + " \\\\"
    header2 = "& " + " & ".join(["$P$ & $R$ & $F1$"] * len(models_present)) + " \\\\"
    lines = []
    for i, row in df.iterrows():
        p = str(row["Pathogen"]).strip()
        left = "\\textbf{Overall}" if p == "Overall" else f"\\textbf{{{p}}}"
        cells = [left]
        for m in models_present:
            for met in METRIC_ORDER:
                v = row.get((m, met), np.nan)
                s = _fmt(v)
                wraps = []
                if met == "F1":
                    if (i, m) in high:
                        wraps.append("\\highP")
                    if (i, m) in low:
                        wraps.append("\\lowP")
                    if (i, m) in best:
                        wraps.append("\\bestM")
                    elif (i, m) in second:
                        wraps.append("\\secondM")
                if s != "" and wraps:
                    s = _wrap(s, wraps[::-1])
                cells.append(s)
        lines.append(" & ".join(cells) + " \\\\")
    midrule_after = keep.index("Overall") if "Overall" in keep else None
    body = []
    for idx, line in enumerate(lines):
        if midrule_after is not None and idx == midrule_after:
            body.append("\\midrule")
        body.append(line)
    body_str = "\n".join(body)
    return (
        "\\begin{table*}[t]\n"
        "\\centering\n"
        f"\\caption{{\\textbf{{{caption}}} \\highP{{Green}} and \\lowP{{Red}} denote the best- and worst-performing pathogens for each model, respectively, also based on F1 score. \\bestM{{Bold}} indicates the best-performing model for each pathogen based on F1 score; \\secondM{{Underline}} indicates the second-best.}}\n"
        f"\\label{{{label}}}\n"
        "\\small\n"
        "\\setlength{\\tabcolsep}{2.6pt}\n"
        "\\renewcommand{\\arraystretch}{1.08}\n"
        "\\resizebox{\\textwidth}{!}{\n"
        f"\\begin{{tabular}}{{{colspec}}}\n"
        "\\toprule\n"
        f"{header1}\n"
        f"{header2}\n"
        "\\midrule\n"
        f"{body_str}\n"
        "\\bottomrule\n"
        "\\end{tabular}\n"
        "}\n"
        "\\end{table*}"
    )


def _high_low_maps_extraction(df):
    models = [m for m in MODEL_ORDER if (m, "F1") in df.columns]
    high = set()
    low = set()
    for kind, g in df.groupby("Kind", dropna=False):
        if str(kind).strip() == "Average":
            continue
        sub = g[~g["Pathogen"].astype(str).str.strip().eq("Overall")].copy()
        if len(sub) == 0:
            continue
        for m in models:
            vals = pd.to_numeric(sub[(m, "F1")], errors="coerce")
            if vals.notna().sum() == 0:
                continue
            mx = vals.max()
            mn = vals.min()
            for idx in sub.index[vals.eq(mx)]:
                high.add((idx, m))
            for idx in sub.index[vals.eq(mn)]:
                low.add((idx, m))
    return high, low, models

def _latex_extraction(df, stage_title, label, caption):
    df = df.copy()
    pathogens = PATHOGENS_BY_STAGE[stage_title]
    df = df[df["Pathogen"].isin(pathogens)].copy()
    p_cat = pd.Categorical(df["Pathogen"], categories=pathogens, ordered=True)
    k_cat = pd.Categorical(df["Kind"], categories=KIND_ORDER, ordered=True)
    df = df.assign(_p=p_cat, _k=k_cat).sort_values(["_p", "_k"]).drop(columns=["_p", "_k"]).reset_index(drop=True)
    models_present = [m for m in MODEL_ORDER if any((m, met) in df.columns for met in METRIC_ORDER)]
    high, low, _ = _high_low_maps_extraction(df)
    best, second, _ = _best_second_model_map(df, by_kind=True)
    colspec = "ll|" + "|".join(["ccc"] * len(models_present))
    header1 = "\\textbf{Pathogen} & \\textbf{Kind} & " + " & ".join(
        [f"\\multicolumn{{3}}{{c|}}{{\\textbf{{{MODEL_NAME_MAP_LATEX[m]}}}}}" for m in models_present[:-1]] +
        ([f"\\multicolumn{{3}}{{c}}{{\\textbf{{{MODEL_NAME_MAP_LATEX[models_present[-1]]}}}}}"] if models_present else [])
    ) + " \\\\"
    header2 = "& & " + " & ".join(["$P$ & $R$ & $F1$"] * len(models_present)) + " \\\\"
    blocks = []
    for p in pathogens:
        sub = df[df["Pathogen"] == p].copy()
        sub = sub[sub["Kind"].isin(KIND_ORDER)].copy()
        sub = sub.assign(_k=pd.Categorical(sub["Kind"], categories=KIND_ORDER, ordered=True)).sort_values("_k").drop(columns=["_k"])
        kinds_present = sub["Kind"].tolist()
        if len(kinds_present) == 0:
            continue
        n = len(kinds_present)
        for j, (_, row) in enumerate(sub.iterrows()):
            if j == 0:
                left = f"\\multirow{{{n}}}{{*}}{{\\textbf{{{p}}}}}"
            else:
                left = ""
            cells = [left, str(row["Kind"]).strip()]
            for m in models_present:
                for met in METRIC_ORDER:
                    v = row.get((m, met), np.nan)
                    s = _fmt(v)
                    wraps = []
                    if met == "F1":
                        if str(row["Kind"]).strip() != "Average":
                            if (row.name, m) in high:
                                wraps.append("\\highP")
                            if (row.name, m) in low:
                                wraps.append("\\lowP")
                        if (row.name, m) in best:
                            wraps.append("\\bestM")
                        elif (row.name, m) in second:
                            wraps.append("\\secondM")
                    if s != "" and wraps:
                        s = _wrap(s, wraps[::-1])
                    cells.append(s)
            blocks.append(" & ".join(cells) + " \\\\")
        if p != pathogens[-1]:
            blocks.append("\\midrule")
    body_str = "\n".join(blocks)
    return (
        "\\begin{table*}[t]\n"
        "\\centering\n"
        f"\\caption{{\\textbf{{{caption}}} \\highP{{Green}} and \\lowP{{Red}} denote the best- and worst-performing pathogens for each model and extraction kind, respectively, also based on F1 score. \\bestM{{Bold}} indicates the best-performing model per row; \\secondM{{Underline}} indicates the second-best.}}\n"
        f"\\label{{{label}}}\n"
        "\\small\n"
        "\\setlength{\\tabcolsep}{2.2pt}\n"
        "\\renewcommand{\\arraystretch}{1.06}\n"
        "\\resizebox{\\textwidth}{!}{\n"
        f"\\begin{{tabular}}{{{colspec}}}\n"
        "\\toprule\n"
        f"{header1}\n"
        f"{header2}\n"
        "\\midrule\n"
        f"{body_str}\n"
        "\\bottomrule\n"
        "\\end{tabular}\n"
        "}\n"
        "\\end{table*}"
    )

def _caption(stage_title):
    if stage_title == "Title and abstract screening":
        return "Title and abstract screening metrics with model ablations."
    if stage_title == "Full-text screening":
        return "Full-text screening metrics with model ablations."
    if stage_title == "Parameter extraction":
        return "Parameter extraction metrics with model ablations."
    if stage_title == "Model extraction":
        return "Model extraction metrics with model ablations."
    if stage_title == "Outbreak extraction":
        return "Outbreak extraction metrics with model ablations."
    return f"{stage_title} metrics with model ablations."

def _label(stage_title):
    k = stage_title.lower().replace("-", " ").replace("/", " ").strip()
    k = "_".join([x for x in k.split() if x])
    if stage_title == "Title and abstract screening":
        return "tab:ta_screening_model_ablation"
    if stage_title == "Full-text screening":
        return "tab:fulltext_screening_model_ablation"
    if stage_title == "Parameter extraction":
        return "tab:parameter_extraction_model_ablation"
    if stage_title == "Model extraction":
        return "tab:model_extraction_model_ablation"
    if stage_title == "Outbreak extraction":
        return "tab:outbreak_extraction_model_ablation"
    return f"tab:{k}_model_ablation"

In [34]:
tables = []

df_abs = _read_screening_wide(ABSTRACT_CSV)
tables.append(_latex_screening(df_abs, "Title and abstract screening", _label("Title and abstract screening"), _caption("Title and abstract screening")))

df_full = _read_screening_wide(FULLTEXT_CSV)
tables.append(_latex_screening(df_full, "Full-text screening", _label("Full-text screening"), _caption("Full-text screening")))

long_params = _read_extraction(PARAMS_CSV)
df_params = _make_extraction_table_df(long_params, PATHOGENS_BY_STAGE["Parameter extraction"])
tables.append(_latex_extraction(df_params, "Parameter extraction", _label("Parameter extraction"), _caption("Parameter extraction")))

long_models = _read_extraction(MODELS_CSV)
df_models = _make_extraction_table_df(long_models, PATHOGENS_BY_STAGE["Model extraction"])
tables.append(_latex_extraction(df_models, "Model extraction", _label("Model extraction"), _caption("Model extraction")))

long_outbreaks = _read_extraction(OUTBREAKS_CSV)
df_outbreaks = _make_extraction_table_df(long_outbreaks, PATHOGENS_BY_STAGE["Outbreak extraction"])
tables.append(_latex_extraction(df_outbreaks, "Outbreak extraction", _label("Outbreak extraction"), _caption("Outbreak extraction")))

print("\n\n".join(tables))

\begin{table*}[t]
\centering
\caption{\textbf{Title and abstract screening metrics with model ablations.} \highP{Green} and \lowP{Red} denote the best- and worst-performing pathogens for each model, respectively, also based on F1 score. \bestM{Bold} indicates the best-performing model for each pathogen based on F1 score; \secondM{Underline} indicates the second-best.}
\label{tab:ta_screening_model_ablation}
\small
\setlength{\tabcolsep}{2.6pt}
\renewcommand{\arraystretch}{1.08}
\resizebox{\textwidth}{!}{
\begin{tabular}{l|ccc|ccc|ccc|ccc|ccc}
\toprule
\textbf{Pathogen} & \multicolumn{3}{c|}{\textbf{GPT-OSS-120B}} & \multicolumn{3}{c|}{\textbf{GPT-5.2}} & \multicolumn{3}{c|}{\textbf{DeepSeek-V3.2}} & \multicolumn{3}{c|}{\textbf{Kimi-K2.5}} & \multicolumn{3}{c}{\textbf{GLM-4.7}} \\
& $P$ & $R$ & $F1$ & $P$ & $R$ & $F1$ & $P$ & $R$ & $F1$ & $P$ & $R$ & $F1$ & $P$ & $R$ & $F1$ \\
\midrule
\textbf{Marburg} & 0.80 & 0.64 & \lowP{\secondM{0.69}} & 0.97 & 0.58 & 0.62 & 0.97 & 0.55 & 0.58 & 0.7

# Tokens Usage, Costing and Performance

In [35]:
import math
import base64
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from pathlib import Path
import cairosvg
import math
import numpy as np
import pandas as pd
import plotly.graph_objects as go


FONT = "Times New Roman, Times, serif"
FONT_SIZE = 24
BG_COLOR = "white"

MODELS_ORDER = ["GPT-OSS-120B (High)", "GPT-5.2 (High)", "DeepSeek-V3.2", "Kimi-K2.5", "GLM-4.7"]

MODEL_COLORS = {
    "GPT-OSS-120B (High)": "#2B2E5F",
    "GLM-4.7":             "#4B4F86",
    "DeepSeek-V3.2":       "#6C74AD",
    "Kimi-K2.5":           "#9AA4CF",
    "GPT-5.2 (High)":      "#BDBDBD"
}

TICK_COLOR = "#000000"
TEXT_COLOR = "rgba(0,0,0,0.88)"

STAGE_COLORS = {
    'Title & Abstract Screening':         '#B7D3C6',
    'Article Screening (AI Conditioned)': '#6BAED6',
    'Parameter Extraction':               '#FFCBA4',
    'Model Extraction':                   '#E38B48',
    'Outbreak Extraction':                '#B45C1A',
}
STAGE_SHORT = {
    'Title & Abstract Screening':         'Title & Abstract',
    'Article Screening (AI Conditioned)': 'Full-text Screening',
    'Parameter Extraction':               'Param. Extraction',
    'Model Extraction':                   'Model Extraction',
    'Outbreak Extraction':                'Outbreak Extraction',
}
MODEL_DISPLAY = {
    'GPT-OSS-120B (High)': 'GPT-OSS-120B',
    'GPT-5.2 (High)':      'GPT-5.2',
    'DeepSeek-V3.2':       'DeepSeek-V3.2',
    'Kimi-K2.5':           'Kimi-K2.5',
    'GLM-4.7':             'GLM-4.7',
}

STAGE_ORDER = ['Title & Abstract Screening','Article Screening (AI Conditioned)',
               'Parameter Extraction','Model Extraction','Outbreak Extraction']


In [36]:
# master_df calculate Average
performance_df = master_df

In [37]:
df_tokens = pd.read_csv("../data/agentslr/evals/stats/token_usage.csv")
df_tokens_avg = df_tokens.groupby(["model", "stage"])[["input_tokens", "output_tokens"]].mean().reset_index()

#### Per-article Numbers

In [38]:
df_tokens_avg

,model,stage,input_tokens,output_tokens
0,deepseek,Article Screening (AI Conditioned),1.285625e+04,837.294788
1,deepseek,Model Extraction,3.724775e+04,3141.173507
2,deepseek,Outbreak Extraction,2.610066e+04,209.509524
3,deepseek,Parameter Extraction,5.232496e+05,2989.410448
4,deepseek,Title & Abstract Screening,2.168332e+03,614.413497
5,glm,Article Screening (AI Conditioned),1.338633e+04,3120.621041
6,glm,Model Extraction,2.973981e+04,1673.203358
7,glm,Outbreak Extraction,2.537823e+04,1980.566667
8,glm,Parameter Extraction,3.050365e+06,32671.718284
9,glm,Title & Abstract Screening,2.224815e+03,1644.283988


#### Average Numbers based on X articles

In [39]:
perg_stage_articles = pd.read_csv("../data/agentslr/evals/stats/articles_passed_through_stages_perg.csv")
perg_stage_articles = perg_stage_articles[perg_stage_articles['Pathogen']=='Average']

titel_and_abstrac_articles_avg = round(perg_stage_articles['Title & Abstract Screening'].values[0])
full_text_screening_articles_avg = round(perg_stage_articles['Full-text Screening'].values[0])
data_extraction_avg = round(perg_stage_articles['Data Extraction'].values[0])

print(f"Average articles passed through Title & Abstract Screening: {titel_and_abstrac_articles_avg}")
print(f"Average articles passed through Full-text Screening: {full_text_screening_articles_avg}")
print(f"Average articles passed through Data Extraction: {data_extraction_avg}")

Average articles passed through Title & Abstract Screening: 9132
Average articles passed through Full-text Screening: 1102
Average articles passed through Data Extraction: 394


In [40]:
df_tokens_avg['input_tokens'] = df_tokens_avg.apply(
    lambda row: row.input_tokens * (
        titel_and_abstrac_articles_avg if row.stage == 'Title & Abstract Screening' else
        full_text_screening_articles_avg if row.stage == 'Article Screening (AI Conditioned)' else
        data_extraction_avg
    ),
    axis=1
)

df_tokens_avg['output_tokens'] = df_tokens_avg.apply(
    lambda row: row.output_tokens * (
        titel_and_abstrac_articles_avg if row.stage == 'Title & Abstract Screening' else
        full_text_screening_articles_avg if row.stage == 'Article Screening (AI Conditioned)' else
        data_extraction_avg
    ),
    axis=1
)

In [41]:
MASTER_DATA = list(
    master_df[['stage', 'model', 'model_csv', 'f1_mean']]
    .itertuples(index=False, name=None)
)

TOKEN_DATA = {
    (row.model, row.stage): (int(row.input_tokens), int(row.output_tokens))
    for row in df_tokens_avg.itertuples(index=False)
}
PRICE_IN  = {'GPT-OSS-120B (High)':0.039,'GPT-5.2 (High)':1.75,'DeepSeek-V3.2':0.28,'Kimi-K2.5':0.6,'GLM-4.7':0.6}
PRICE_OUT = {'GPT-OSS-120B (High)':0.19,'GPT-5.2 (High)':14.0,'DeepSeek-V3.2':0.42,'Kimi-K2.5':3.0,'GLM-4.7':2.2}

In [42]:
rows = []
for stage, model, csv_key, f1 in MASTER_DATA:
    tok_key = (csv_key, stage)
    if tok_key not in TOKEN_DATA:
        continue
    in_tok, out_tok = TOKEN_DATA[tok_key]
    cost = (in_tok/1e6)*PRICE_IN[model] + (out_tok/1e6)*PRICE_OUT[model]
    rows.append({'stage':stage,'model':model,'csv':csv_key,'f1':f1,'cost':cost})

df = pd.DataFrame(rows)

In [43]:
df = df.copy()
df["cost"] = df["cost"].astype(float)
df["f1"] = df["f1"].astype(float)

sum_df = (
    df.groupby("model", as_index=False)
    .agg(cost=("cost", "sum"), f1=("f1", "mean"))
)

# IMPORTANT: SD across STAGES for each model (stage-level deviation)
err_df = (
    df.groupby("model", as_index=False)
    .agg(cost_err=("cost", "std"), f1_err=("f1", "std"))
)

sum_df = sum_df.merge(err_df, on="model", how="left")
sum_df["cost_err"] = sum_df["cost_err"].fillna(0.0)
sum_df["f1_err"] = sum_df["f1_err"].fillna(0.0)

sum_df_plot = sum_df[sum_df["model"].isin(MODELS_ORDER)].copy()
sum_df_plot["model"] = pd.Categorical(sum_df_plot["model"], categories=MODELS_ORDER, ordered=True)
sum_df_plot = sum_df_plot.sort_values("model")

In [44]:
sum_df_plot[['model', 'cost', 'f1', 'f1_err']].to_dict(orient='records')

[{'model': 'GPT-OSS-120B (High)',
  'cost': 13.936967133,
  'f1': 0.6953147000546008,
  'f1_err': 0.07167016546353597},
 {'model': 'GPT-5.2 (High)',
  'cost': 1348.1950405,
  'f1': 0.6906476530778156,
  'f1_err': 0.09179737980160195},
 {'model': 'DeepSeek-V3.2',
  'cost': 73.6635375,
  'f1': 0.6784085304678955,
  'f1_err': 0.11353394271583492},
 {'model': 'Kimi-K2.5',
  'cost': 277.17506039999995,
  'f1': 0.7427758280418897,
  'f1_err': 0.07562359685974866},
 {'model': 'GLM-4.7',
  'cost': 810.8475625999998,
  'f1': 0.7289453828657521,
  'f1_err': 0.09040297419525184}]

In [45]:
def fmt_dollar(x):
    if x >= 1000:
        return f"${x/1000:.0f}k"
    if x >= 1:
        return f"${x:.0f}"
    if x >= 0.01:
        return f"${x:.2f}"
    return f"${x:.3f}"

def build_log_ticks(xmin, xmax):
    kmin = int(math.floor(math.log10(xmin)))
    kmax = int(math.ceil(math.log10(xmax)))
    vals = []
    for k in range(kmin, kmax + 1):
        base = 10 ** k
        for m in (1, 2, 5):
            v = m * base
            if xmin <= v <= xmax:
                vals.append(v)
    vals = sorted(set(vals))
    return vals, [fmt_dollar(v) for v in vals]

def build_cost_vs_accuracy_fig(
    df,
    out_pdf_path="cost_vs_accuracy.pdf",
    out_svg_path="cost_vs_accuracy.svg",
):
    df = df.copy()
    df["cost"] = df["cost"].astype(float)
    df["f1"] = df["f1"].astype(float)

    sum_df = (
        df.groupby("model", as_index=False)
        .agg(cost=("cost", "sum"), f1=("f1", "mean"))
    )

    # IMPORTANT: SD across STAGES for each model (stage-level deviation)
    err_df = (
        df.groupby("model", as_index=False)
        .agg(cost_err=("cost", "std"), f1_err=("f1", "std"))
    )

    sum_df = sum_df.merge(err_df, on="model", how="left")
    sum_df["cost_err"] = sum_df["cost_err"].fillna(0.0)
    sum_df["f1_err"] = sum_df["f1_err"].fillna(0.0)

    sum_df_plot = sum_df[sum_df["model"].isin(MODELS_ORDER)].copy()
    sum_df_plot["model"] = pd.Categorical(sum_df_plot["model"], categories=MODELS_ORDER, ordered=True)
    sum_df_plot = sum_df_plot.sort_values("model")

    eps = 1e-12

    cx = sum_df_plot["cost"].to_numpy(dtype=float)
    cy = sum_df_plot["f1"].to_numpy(dtype=float)
    ex = sum_df_plot["cost_err"].to_numpy(dtype=float)
    ey = sum_df_plot["f1_err"].to_numpy(dtype=float)

    ex_minus = np.minimum(ex, np.maximum(cx - eps, 0.0))
    ex_plus = ex
    x_low = np.maximum(cx - ex_minus, eps)
    x_high = cx + ex_plus
    y_low = cy - ey
    y_high = cy + ey

    xmin = float(np.min(x_low))
    xmax = float(np.max(x_high))
    ymin = float(np.min(y_low))
    ymax = float(np.max(y_high))

    x_pad = (math.log10(xmax) - math.log10(xmin)) * 0.08 if xmax > xmin else 0.3
    log_xmin = math.log10(xmin) - x_pad
    log_xmax = math.log10(xmax) + x_pad

    y_pad = (ymax - ymin) * 0.10 if ymax > ymin else 0.02
    ymin = ymin - y_pad
    ymax = ymax + y_pad

    tick_vals, tick_text = build_log_ticks(xmin, xmax)

    fig = go.Figure()

    AXIS_LINECOLOR = "#BDBDBD"
    AXIS_LINEWIDTH = 1
    TICKLEN = 7
    TICKWIDTH = 1
    TITLE_STANDOFF = 12
    TICK_STANDOFF = 6

    label_dy = (ymax - ymin) * 0.035
    dot_dy = (ymax - ymin) * 0.012
    cap_dy = (ymax - ymin) * 0.008
    # ERR_COLOR = "#7f7f7f"
    ERR_W = 8
    CAP_W = 5

    for _, r in sum_df_plot.iterrows():
        model = str(r["model"])
        c = MODEL_COLORS[model]
        x = float(r["cost"])
        y = float(r["f1"])
        dx = float(r["cost_err"])
        dy = float(r["f1_err"])

        dxm = float(min(dx, max(x - eps, 0.0)))
        dxp = dx
        dym = dy
        dyp = dy

        x0 = max(x - dxm, eps)
        x1 = x + dxp
        y0 = y - dym
        y1 = y + dyp

        if dy > 0:
            fig.add_shape(type="line", xref="x", yref="y", x0=x, y0=y0, x1=x, y1=y1,
                          line=dict(color=c, width=ERR_W), layer="above")
            xratio = 1.06
            fig.add_shape(type="line", xref="x", yref="y", x0=x/xratio, y0=y0, x1=x*xratio, y1=y0,
                          line=dict(color=c, width=CAP_W), layer="above")
            fig.add_shape(type="line", xref="x", yref="y", x0=x/xratio, y0=y1, x1=x*xratio, y1=y1,
                          line=dict(color=c, width=CAP_W), layer="above")

        # main cross
        fig.add_trace(
            go.Scatter(
                x=[x],
                y=[y],
                mode="markers",
                marker=dict(symbol="circle", size=22, color=c, line=dict(width=2, color=c)),
                hoverinfo="skip",
                showlegend=False,
            )
        )

        # label + dot under label
        fig.add_trace(
                go.Scatter(
                    x=[x],
                    y=[0.83],
                    mode="text",
                    text=[MODEL_DISPLAY.get(model, model)],
                    textposition="middle center",
                    textfont=dict(family=FONT, size=FONT_SIZE, color=TEXT_COLOR),
                    hoverinfo="skip",
                    showlegend=False,
                )
            )

    fig.update_xaxes(
        type="log",
        range=[log_xmin, log_xmax],
        tickmode="array",
        tickvals=tick_vals,
        ticktext=tick_text,
        title_text="Average Cost per Pathogen run (log<sub>10</sub> USD)",
        title_font=dict(family=FONT, size=FONT_SIZE, color=TICK_COLOR),
        tickfont=dict(family=FONT, size=FONT_SIZE, color=TICK_COLOR),
        showgrid=False,
        zeroline=False,
        showline=True,
        linecolor=AXIS_LINECOLOR,
        linewidth=AXIS_LINEWIDTH,
        ticks="outside",
        ticklen=TICKLEN,
        tickwidth=TICKWIDTH,
        tickcolor=TICK_COLOR,
        ticklabelposition="outside",
        ticklabelstandoff=TICK_STANDOFF,
        title_standoff=TITLE_STANDOFF,
        mirror=False,
    )

    fig.update_yaxes(
        range=[ymin, ymax],
        title_text="Performance (F<sub>1</sub> Score)",
        title_font=dict(family=FONT, size=FONT_SIZE, color=TICK_COLOR),
        tickfont=dict(family=FONT, size=FONT_SIZE, color=TICK_COLOR),
        showgrid=False,
        zeroline=False,
        showline=True,
        linecolor=AXIS_LINECOLOR,
        linewidth=AXIS_LINEWIDTH,
        ticks="outside",
        ticklen=TICKLEN,
        tickwidth=TICKWIDTH,
        tickcolor=TICK_COLOR,
        ticklabelposition="outside",
        ticklabelstandoff=TICK_STANDOFF,
        title_standoff=TITLE_STANDOFF,
        mirror=False,
    )

    fig.update_layout(
        width=1100,
        height=680,
        font=dict(family=FONT, size=FONT_SIZE, color=TICK_COLOR),
        plot_bgcolor=BG_COLOR,
        paper_bgcolor=BG_COLOR,
        margin=dict(t=95, b=80, l=95, r=30),
    )

    # fig.write_image(out_pdf_path, scale=1)
    fig.write_image(out_svg_path, scale=1)
    return fig


fig = build_cost_vs_accuracy_fig(
    df=df,
    out_pdf_path="../assets/figures/cost_vs_accuracy/cost_vs_accuracy_raw.pdf",
    out_svg_path="../assets/figures/cost_vs_accuracy/cost_vs_accuracy_raw.svg",
)

fig.show()

## Appendix Table and Fig

In [46]:

df_tokens_avg = pd.read_csv("../data/agentslr/evals/stats/token_usage.csv").groupby(["model", "stage"])[["input_tokens", "output_tokens"]].mean().reset_index()


MODEL_FROM_CSV = {
    "oss": "GPT-OSS-120B (High)",
    "gpt": "GPT-5.2 (High)",
    "deepseek": "DeepSeek-V3.2",
    "kimi": "Kimi-K2.5",
    "glm": "GLM-4.7",
}

MODELS_ORDER = ["GPT-OSS-120B (High)", "GPT-5.2 (High)", "DeepSeek-V3.2", "Kimi-K2.5", "GLM-4.7"]
STAGE_ORDER = [
    "Title & Abstract Screening",
    "Article Screening (AI Conditioned)",
    "Parameter Extraction",
    "Model Extraction",
    "Outbreak Extraction",
]
SUBCOLS = ["Input Tok", "Output Tok", "Cost"]

df_tok = df_tokens_avg.copy()
df_tok["model_full"] = df_tok["model"].map(MODEL_FROM_CSV)
df_tok = df_tok[df_tok["model_full"].isin(MODELS_ORDER) & df_tok["stage"].isin(STAGE_ORDER)]

df_tok["Input Tok"] = df_tok["input_tokens"].astype(float)
df_tok["Output Tok"] = df_tok["output_tokens"].astype(float)

df_tok["Cost"] = (
    (df_tok["Input Tok"] / 1e6) * df_tok["model_full"].map(PRICE_IN)
    + (df_tok["Output Tok"] / 1e6) * df_tok["model_full"].map(PRICE_OUT)
)

df_tok["stage"] = pd.Categorical(df_tok["stage"], categories=STAGE_ORDER, ordered=True)
df_tok["model_full"] = pd.Categorical(df_tok["model_full"], categories=MODELS_ORDER, ordered=True)

master_tokens_df = (
    df_tok.pivot_table(
        index="stage",
        columns="model_full",
        values=["Input Tok", "Output Tok", "Cost"],
        aggfunc="first",
    )
    .swaplevel(0, 1, axis=1)
    .sort_index(axis=1, level=0)
    .reindex(columns=pd.MultiIndex.from_product([MODELS_ORDER, SUBCOLS]), fill_value=np.nan)
)

master_tokens_df = master_tokens_df.astype(float)

overall_row = master_tokens_df.sum(axis=0)
master_tokens_df.loc["Overall"] = overall_row

master_tokens_df = master_tokens_df.reindex(STAGE_ORDER + ["Overall"])

In [51]:
import math
import numpy as np
import pandas as pd

def _format_tokens_k(x):
    if pd.isna(x):
        return "--"
    return f"{float(x)/1000.0:.1f} K"

def _format_cost_cut(x):
    if pd.isna(x):
        return "--"
    x = float(x)
    if x > 0 and x < 0.01:
        return r"$<0.01$"
    v = math.floor(x * 100.0) / 100.0
    return f"{v:.2f}"

def _wrap_stage(stage):
    m = {
        "Title & Abstract Screening": r"\makecell[l]{Title \& Abstract\\Screening}",
        "Article Screening (AI Conditioned)": r"\makecell[l]{Article Screening\\(AI Conditioned)}",
        "Parameter Extraction": r"\makecell[l]{Parameter\\Extraction}",
        "Model Extraction": r"\makecell[l]{Model\\Extraction}",
        "Outbreak Extraction": r"\makecell[l]{Outbreak\\Extraction}",
        "Overall": r"\textbf{Overall}",
    }
    return m.get(stage, stage.replace("&", r"\&"))

def make_token_usage_latex_table(
    master_tokens_df: pd.DataFrame,
    models_order,
    subcols=("Input Tok", "Output Tok", "Cost"),
    stage_order=None,
    caption=r"\textbf{Token usage and cost by stage.} \highP{Green} marks the lowest (best) usage/cost and \lowP{Red} marks the highest (worst) usage/cost per row and subcolumn.",
    label="tab:token_usage_by_stage",
):
    df = master_tokens_df.copy()
    if stage_order is not None:
        df = df.reindex(stage_order)

    def wrap_val(val_str, is_best, is_worst):
        if is_best and is_worst:
            return rf"\highP{{\lowP{{{val_str}}}}}"
        if is_best:
            return rf"\highP{{{val_str}}}"
        if is_worst:
            return rf"\lowP{{{val_str}}}"
        return val_str

    header_models = " & ".join(
        [rf"\multicolumn{{3}}{{c{'|' if i < len(models_order)-1 else ''}}}{{\textbf{{{m}}}}}"
         for i, m in enumerate(models_order)]
    )
    header_sub = " & ".join(
        [r"\makecell[c]{Input\\Tokens} & \makecell[c]{Output\\Tokens} & \makecell[c]{Cost\\(USD Paper)}"
         for _ in models_order]
    )

    colspec = "l|" + "|".join(["ccc" for _ in models_order])

    lines = []
    lines.append(r"\begin{table*}[h]")
    lines.append(r"\centering")
    lines.append(rf"\caption{{{caption}}}")
    lines.append(rf"\label{{{label}}}")
    lines.append(r"\small")
    lines.append(r"\setlength{\tabcolsep}{2.2pt}")
    lines.append(r"\renewcommand{\arraystretch}{1.06}")
    lines.append(r"\resizebox{\textwidth}{!}{")
    lines.append(rf"\begin{{tabular}}{{{colspec}}}")
    lines.append(r"\toprule")
    lines.append(rf"\textbf{{Stage}} & {header_models} \\")
    lines.append(rf" & {header_sub} \\")
    lines.append(r"\midrule")

    for stage in df.index:
        row_parts = [_wrap_stage(stage)]

        mins = {}
        maxs = {}
        for sub in subcols:
            vals = []
            for model in models_order:
                v = df.loc[stage, (model, sub)] if (model, sub) in df.columns else np.nan
                vals.append(v)
            arr = np.array(vals, dtype=float)
            finite = np.isfinite(arr)
            mins[sub] = np.min(arr[finite]) if finite.any() else np.nan
            maxs[sub] = np.max(arr[finite]) if finite.any() else np.nan

        for model in models_order:
            for sub in subcols:
                v = df.loc[stage, (model, sub)] if (model, sub) in df.columns else np.nan
                if sub == "Cost":
                    s = _format_cost_cut(v)
                else:
                    s = _format_tokens_k(v)
                is_best = (np.isfinite(v) and np.isfinite(mins[sub]) and float(v) == float(mins[sub]))
                is_worst = (np.isfinite(v) and np.isfinite(maxs[sub]) and float(v) == float(maxs[sub]))
                row_parts.append(wrap_val(s, is_best, is_worst))

        lines.append(" & ".join(row_parts) + r" \\")
        if stage != df.index[-1]:
            lines.append(r"\midrule")

    lines.append(r"\bottomrule")
    lines.append(r"\end{tabular}")
    lines.append(r"}")
    lines.append(r"\end{table*}")
    return "\n".join(lines)

latex = make_token_usage_latex_table(
    master_tokens_df=master_tokens_df,
    models_order=MODELS_ORDER,
    subcols=("Input Tok", "Output Tok", "Cost"),
    stage_order=STAGE_ORDER + ["Overall"],
)
print(latex)

\begin{table*}[h]
\centering
\caption{\textbf{Token usage and cost by stage.} \highP{Green} marks the lowest (best) usage/cost and \lowP{Red} marks the highest (worst) usage/cost per row and subcolumn.}
\label{tab:token_usage_by_stage}
\small
\setlength{\tabcolsep}{2.2pt}
\renewcommand{\arraystretch}{1.06}
\resizebox{\textwidth}{!}{
\begin{tabular}{l|ccc|ccc|ccc|ccc|ccc}
\toprule
\textbf{Stage} & \multicolumn{3}{c|}{\textbf{GPT-OSS-120B (High)}} & \multicolumn{3}{c|}{\textbf{GPT-5.2 (High)}} & \multicolumn{3}{c|}{\textbf{DeepSeek-V3.2}} & \multicolumn{3}{c|}{\textbf{Kimi-K2.5}} & \multicolumn{3}{c}{\textbf{GLM-4.7}} \\
 & \makecell[c]{Input\\Tokens} & \makecell[c]{Output\\Tokens} & \makecell[c]{Cost\\(USD Paper)} & \makecell[c]{Input\\Tokens} & \makecell[c]{Output\\Tokens} & \makecell[c]{Cost\\(USD Paper)} & \makecell[c]{Input\\Tokens} & \makecell[c]{Output\\Tokens} & \makecell[c]{Cost\\(USD Paper)} & \makecell[c]{Input\\Tokens} & \makecell[c]{Output\\Tokens} & \makecell[c]{Cost\\(USD 

In [50]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

FONT = "Times New Roman, Times, serif"
FONT_SIZE = 21
BG_COLOR = "white"

STAGE_COLORS = {
    "Title & Abstract Screening": "#B7D3C6",
    "Article Screening (AI Conditioned)": "#6BAED6",
    "Parameter Extraction": "#FFCBA4",
    "Model Extraction": "#E38B48",
    "Outbreak Extraction": "#B45C1A",
}
STAGE_SHORT = {
    "Title & Abstract Screening": "Title & Abstract",
    "Article Screening (AI Conditioned)": "Full-text Screening",
    "Parameter Extraction": "Param. Extraction",
    "Model Extraction": "Model Extraction",
    "Outbreak Extraction": "Outbreak Extraction",
}
STAGE_ORDER = [
    "Title & Abstract Screening",
    "Article Screening (AI Conditioned)",
    "Parameter Extraction",
    "Model Extraction",
    "Outbreak Extraction",
]

MODEL_FROM_CSV = {
    "oss": "GPT-OSS-120B (High)",
    "gpt": "GPT-5.2 (High)",
    "deepseek": "DeepSeek-V3.2",
    "kimi": "Kimi-K2.5",
    "glm": "GLM-4.7",
}

PRICE_IN = {"GPT-OSS-120B (High)": 0.039, "GPT-5.2 (High)": 1.75, "DeepSeek-V3.2": 0.28, "Kimi-K2.5": 0.6, "GLM-4.7": 0.6}
PRICE_OUT = {"GPT-OSS-120B (High)": 0.19, "GPT-5.2 (High)": 14.0, "DeepSeek-V3.2": 0.42, "Kimi-K2.5": 3.0, "GLM-4.7": 2.2}

df_tokens_usage = pd.read_csv("../data/agentslr/evals/stats/token_usage.csv")
df_tokens_usage = df_tokens_usage.groupby(["model", "stage"])[["input_tokens", "output_tokens"]].mean().reset_index()

perg_stage_articles = pd.read_csv("../data/agentslr/evals/stats/articles_passed_through_stages_perg.csv")
perg_stage_articles = perg_stage_articles[perg_stage_articles["Pathogen"] == "Average"]

n_ta = int(round(perg_stage_articles["Title & Abstract Screening"].values[0]))
n_ft = int(round(perg_stage_articles["Full-text Screening"].values[0]))
n_de = int(round(perg_stage_articles["Data Extraction"].values[0]))

articles_map = {
    "Title & Abstract Screening": float(n_ta),
    "Article Screening (AI Conditioned)": float(n_ft),
    "Parameter Extraction": float(n_de),
    "Model Extraction": float(n_de),
    "Outbreak Extraction": float(n_de),
}

df = df_tokens_usage.copy()
df["model_full"] = df["model"].map(MODEL_FROM_CSV)
df = df[df["model_full"].notna() & df["stage"].isin(STAGE_ORDER)].copy()

mult = np.array([articles_map[s] for s in df["stage"].values], dtype=float)
df["input_tokens_scaled"] = df["input_tokens"].astype(float) * mult
df["output_tokens_scaled"] = df["output_tokens"].astype(float) * mult
df["cost"] = (df["input_tokens_scaled"] / 1e6) * df["model_full"].map(PRICE_IN).astype(float) + (df["output_tokens_scaled"] / 1e6) * df["model_full"].map(PRICE_OUT).astype(float)

stage_avg = df.groupby("stage", as_index=False).agg(
    input_tokens=("input_tokens_scaled", "mean"),
    output_tokens=("output_tokens_scaled", "mean"),
    cost=("cost", "mean"),
)
stage_avg["articles"] = stage_avg["stage"].map(articles_map).astype(float)

stage_avg["stage"] = pd.Categorical(stage_avg["stage"], categories=STAGE_ORDER, ordered=True)
stage_avg = stage_avg.sort_values("stage")

stages = stage_avg["stage"].astype(str).tolist()
x = [STAGE_SHORT[s] for s in stages]
colors = [STAGE_COLORS[s] for s in stages]

y_articles = stage_avg["articles"].astype(float).values
y_in = stage_avg["input_tokens"].astype(float).values
y_out = stage_avg["output_tokens"].astype(float).values
y_cost = stage_avg["cost"].astype(float).values

def fmt_int(x):
    return f"{int(round(float(x))):,}"

def fmt_tok(x):
    x = float(x)
    if x >= 1e9:
        return f"{x/1e9:.2f}B"
    if x >= 1e6:
        return f"{x/1e6:.2f}M"
    return f"{x/1e3:.2f}K"

def fmt_cost(x):
    x = float(x)
    if x >= 1000:
        return f"${x/1000:.0f}k"
    if x >= 1:
        return f"${x:.0f}"
    if x >= 0.01:
        return f"${x:.2f}"
    return f"${x:.3f}"

fig = make_subplots(
    rows=1,
    cols=4,
    horizontal_spacing=0.04,
    # subplot_titles=("Articles", "Input Tokens", "Output Tokens", "Cost (USD)"),
)

bar_w = 0.72
textpos = "outside"

for i, (title, y, fmt) in enumerate(
    [
        ("Articles", y_articles, fmt_int),
        ("Input Tokens", y_in, fmt_tok),
        ("Output Tokens", y_out, fmt_tok),
        ("Cost (USD)", y_cost, fmt_cost),
    ],
    start=1,
):
    for xi, yi, c, st in zip(x, y, colors, stages):
        fig.add_trace(
            go.Bar(
                x=[xi],
                y=[yi],
                width=bar_w,
                marker=dict(color=c, line=dict(width=0)),
                name=STAGE_SHORT[st],
                showlegend=(i == 1),
                text=[fmt(yi)],
                textposition=textpos,
                cliponaxis=False,
                hoverinfo="skip",
            ),
            row=1,
            col=i,
        )

common_axis = dict(
    showgrid=False,
    zeroline=False,
    showline=True,
    linecolor="#BDBDBD",
    linewidth=1,
)

for c in [1, 2, 3, 4]:
    fig.update_xaxes(showticklabels=False, ticks="", showgrid=False, zeroline=False, showline=True, linecolor="#BDBDBD", linewidth=1, row=1, col=c)
    fig.update_yaxes(
        showgrid=False,
        zeroline=False,
        showline=True,
        linecolor="#BDBDBD",
        linewidth=1,
        ticks="outside",
        ticklen=7,
        tickwidth=1,
        tickcolor="#111111",
        tickfont=dict(family=FONT, size=FONT_SIZE, color="#111111"),
        row=1,
        col=c,
    )

fig.update_yaxes(title_text="", row=1, col=1)
fig.update_yaxes(title_text="", row=1, col=2)
fig.update_yaxes(title_text="", row=1, col=3)
fig.update_yaxes(title_text="", row=1, col=4)

fig.update_annotations(font=dict(family=FONT, size=FONT_SIZE, color="#111111"))

fig.update_layout(
    width=1500,
    height=420,
    margin=dict(t=70, b=42, l=55, r=18),
    font=dict(family=FONT, size=FONT_SIZE, color="#111111"),
    plot_bgcolor=BG_COLOR,
    paper_bgcolor=BG_COLOR,
    barmode="group",
    bargap=0.06,
    bargroupgap=0.02,
    barcornerradius=10,
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.06,
        xanchor="center",
        x=0.5,
        font=dict(family=FONT, size=FONT_SIZE, color="#111111"),
        bgcolor="rgba(255,255,255,0)",
        borderwidth=0,
        itemsizing="constant",
        traceorder="normal",
    ),
)

bottom_titles = ["Articles", "Input Tokens", "Output Tokens", "Cost (USD)"]

for i, title in enumerate(bottom_titles, start=1):
    dom = fig.layout[f"xaxis{i}"].domain
    fig.add_annotation(
        text=title,
        x=(dom[0] + dom[1]) / 2,
        y=-0.10,
        xref="paper",
        yref="paper",
        showarrow=False,
        xanchor="center",
        font=dict(family=FONT, size=FONT_SIZE, color="#111111"),
    )
fig.write_image("../assets/figures/token_articles_cost_by_stage.pdf", scale=1)
fig.show()


# Extra — Model Evals Bars for Fig. 1.

In [ ]:
import plotly.graph_objects as go

vals = [0.846, 0.812, 0.806, 0.767, 0.751]
errs = [0.045, 0.040, 0.038, 0.036, 0.034]
models = ["GPT-OSS-120B (High)", "DeepSeek-V3.2", "Kimi-K2.5", "GPT-5.2 (High)", "GLM-4.7"]

ymin, ymax = 0.0, 1.0
layout_h = 480
layout_w = 620
margin_t, margin_b = 6, 6

BAR_CORNER_RADIUS = 18

FONT_SIZE = 20

fig = go.Figure()

fig.add_trace(
    go.Bar(
        x=models,
        y=vals,
        marker=dict(
            color=[MODEL_COLORS[m] for m in models],
            line=dict(width=0),
            cornerradius=BAR_CORNER_RADIUS,
        ),
        error_y=dict(
            type="data",
            array=errs,
            symmetric=True,
            thickness=ERR_THICKNESS,
            width=ERR_CAP,
            color=ERR_COLOR,
        ),
        showlegend=False,
        hoverinfo="skip",
    )
)

# for m, v, e in zip(models, vals, errs):
#     fig.add_annotation(
#         x=m,
#         y=min(ymax, v + e + 0.025),
#         text=f"{v*100:.1f}",
#         showarrow=False,
#         font=dict(family=FONT, size=FONT_SIZE, color="#111111"),
#         xref="x", yref="y",
#     )

for m, v, e in zip(models, vals, errs):
    icon_src = ICON_SOURCES[model_to_icon_key(m)]
    sizey = min(0.30, v * 0.48)
    sizex = 0.52
    y_center = max(0.55 * v, v - sizey * 0.55)

    fig.add_layout_image(
        dict(
            source=icon_src,
            xref="x", yref="y",
            x=m, y=y_center,
            xanchor="center", yanchor="middle",
            sizex=sizex, sizey=sizey,
            sizing="contain", layer="above",
        )
    )

fig.update_layout(
    height=layout_h,
    width=layout_w,
    plot_bgcolor=BG_COLOR,
    paper_bgcolor=BG_COLOR,
    margin=dict(t=margin_t, b=margin_b, l=6, r=6),
    showlegend=False,
    bargap=0.04,
    barmode="overlay",
    barcornerradius=BAR_CORNER_RADIUS,
)

fig.update_xaxes(showgrid=False, zeroline=False, showticklabels=False, showline=False)
fig.update_yaxes(showgrid=False, zeroline=False, showticklabels=False, showline=False, range=[ymin, ymax])

fig.write_image("../assets/five_bars.svg")
fig.show()